# 1\.1\.7 Load Weather Observations

This notebook stages hourly weather observations as a contextual sidecar for the App Prep NYC congestion pricing project\. The job here is to load the station\-level weather source cleanly, document what we kept, and leave behind a study\-period hourly table that downstream notebooks can trust\. Weather is still intentionally scoped as context rather than a core mobility modality, so this notebook stops at the station\-hour handoff and does not try to assign weather to Taxi Zones or fold it into the merged mobility layer\.

Before loading any weather data, we need to lock down the notebook boundaries and make the execution state explicit\. This setup block defines the study\-period constants, output paths, rebuild toggles, and a few small helpers so the rest of the notebook can run predictably without relying on hidden notebook memory\. It also keeps the weather workflow aligned with the rest of the project: explicit paths, conservative toggles, and enough progress/output structure to make pilot runs easy to inspect\.

In [1]:
from __future__ import annotations

import os
import time
from typing import Dict, List, Optional
from pathlib import Path

import pandas as pd
import requests

from IPython.display import display

# Show full strings in pandas output
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

In [2]:
# ---------------------------------------------------------------------
# 1.1.7.0 Setup: constants, paths, toggles, and lightweight helpers
# ---------------------------------------------------------------------

STUDY_START_DATE = "2023-01-01"
STUDY_END_DATE = "2026-03-31"
CONGESTION_PRICING_START_DATE = "2025-01-05"
PROJECT_TIMEZONE = "America/New_York"

PIPELINE_DATA_DIR = Path("pipeline_data")
SOURCE_DATA_DIR = Path("source_data")

WEATHER_INTERMEDIATE_DIR = PIPELINE_DATA_DIR / "1.1.7.intermediate_tables"
WEATHER_STAGED_DIR = PIPELINE_DATA_DIR / "1.1.7.staged_weather"
WEATHER_QA_DIR = PIPELINE_DATA_DIR / "1.1.7.qa"
WEATHER_MANIFEST_DIR = PIPELINE_DATA_DIR / "1.1.7.manifests"
WEATHER_RAW_SOURCE_DIR = SOURCE_DATA_DIR / "weather" / "noaa" / "ghcnh"
WEATHER_RAW_DOWNLOAD_DIR = WEATHER_RAW_SOURCE_DIR / "station_year"

for path in [
    WEATHER_INTERMEDIATE_DIR,
    WEATHER_STAGED_DIR,
    WEATHER_QA_DIR,
    WEATHER_MANIFEST_DIR,
    WEATHER_RAW_DOWNLOAD_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

RUN_PILOT_LOAD = True
RUN_FULL_WEATHER_LOAD = False
DOWNLOAD_PILOT_WEATHER_SOURCE = False
FORCE_REBUILD_WEATHER_OUTPUTS = False
SAVE_WEATHER_OUTPUTS = True
WRITE_FULL_WEATHER_AUDIT_EXPORT = False
CACHE_PILOT_RAW_WEATHER = False

NOAA_API_TOKEN_ENV_VAR = "NOAA_API_TOKEN"
NOAA_API_BASE_URL = "https://www.ncei.noaa.gov/cdo-web/api/v2"
NOAA_DATASET_ID = "LCD"
NOAA_REQUEST_LIMIT = 1000

WEATHER_PILOT_RAW_CACHE_PATH = WEATHER_INTERMEDIATE_DIR / "weather_pilot_raw.parquet"
WEATHER_PILOT_STANDARDIZED_PATH = WEATHER_INTERMEDIATE_DIR / "weather_pilot_standardized.parquet"
WEATHER_HOURLY_OUTPUT_PATH = WEATHER_STAGED_DIR / "weather_hourly_observations.parquet"
WEATHER_HOURLY_AUDIT_OUTPUT_PATH = WEATHER_STAGED_DIR / "weather_hourly_observations_full_audit.parquet"
WEATHER_STATION_MANIFEST_PATH = WEATHER_STAGED_DIR / "weather_station_manifest.parquet"
WEATHER_HOURLY_QA_SUMMARY_PATH = WEATHER_QA_DIR / "weather_hourly_qa_summary.parquet"
WEATHER_DATA_CONTRACT_PATH = WEATHER_MANIFEST_DIR / "weather_data_contract.md"

STUDY_START_TIMESTAMP = pd.Timestamp(STUDY_START_DATE)
STUDY_END_TIMESTAMP_EXCLUSIVE = pd.Timestamp(STUDY_END_DATE) + pd.Timedelta(days=1)
CONGESTION_PRICING_START_TIMESTAMP = pd.Timestamp(CONGESTION_PRICING_START_DATE)


def print_step(message: str) -> None:
    """Emit compact progress messages for notebook execution."""
    print(f"[1.1.7] {message}", flush=True)



def should_build(output_path: Path, force_rebuild: bool = False) -> bool:
    """Return True when an output should be rebuilt."""
    return force_rebuild or not output_path.exists()



def describe_output(output_path: Path) -> str:
    """Human-readable status for expected notebook outputs."""
    return "EXISTS" if output_path.exists() else "MISSING"


print_step("Weather notebook setup loaded.")

setup_summary_df = pd.DataFrame(
    {
        "setting": [
            "study_start_date",
            "study_end_date",
            "congestion_pricing_start_date",
            "project_timezone",
            "run_pilot_load",
            "run_full_weather_load",
            "download_pilot_weather_source",
            "force_rebuild_weather_outputs",
            "save_weather_outputs",
            "write_full_weather_audit_export",
            "cache_pilot_raw_weather",
            "noaa_api_token_env_var",
            "noaa_api_base_url",
            "noaa_dataset_id",
            "pilot_raw_cache_path",
            "pilot_standardized_path",
            "weather_hourly_output_path",
            "weather_hourly_audit_output_path",
            "weather_station_manifest_path",
            "weather_hourly_qa_summary_path",
            "weather_data_contract_path",
            "weather_raw_source_dir",
            "weather_raw_download_dir",
        ],
        "value": [
            STUDY_START_DATE,
            STUDY_END_DATE,
            CONGESTION_PRICING_START_DATE,
            PROJECT_TIMEZONE,
            RUN_PILOT_LOAD,
            RUN_FULL_WEATHER_LOAD,
            DOWNLOAD_PILOT_WEATHER_SOURCE,
            FORCE_REBUILD_WEATHER_OUTPUTS,
            SAVE_WEATHER_OUTPUTS,
            WRITE_FULL_WEATHER_AUDIT_EXPORT,
            CACHE_PILOT_RAW_WEATHER,
            NOAA_API_TOKEN_ENV_VAR,
            NOAA_API_BASE_URL,
            NOAA_DATASET_ID,
            str(WEATHER_PILOT_RAW_CACHE_PATH),
            str(WEATHER_PILOT_STANDARDIZED_PATH),
            str(WEATHER_HOURLY_OUTPUT_PATH),
            str(WEATHER_HOURLY_AUDIT_OUTPUT_PATH),
            str(WEATHER_STATION_MANIFEST_PATH),
            str(WEATHER_HOURLY_QA_SUMMARY_PATH),
            str(WEATHER_DATA_CONTRACT_PATH),
            str(WEATHER_RAW_SOURCE_DIR),
            str(WEATHER_RAW_DOWNLOAD_DIR),
        ],
    }
)

display(setup_summary_df)

[1.1.7] Weather notebook setup loaded.


,setting,value
0,study_start_date,2023-01-01
1,study_end_date,2026-03-31
2,congestion_pricing_start_date,2025-01-05
3,project_timezone,America/New_York
4,run_pilot_load,True
5,run_full_weather_load,False
6,download_pilot_weather_source,False
7,force_rebuild_weather_outputs,False
8,save_weather_outputs,True
9,write_full_weather_audit_export,False


Findings\. The setup block is doing exactly what we want at this stage: the study\-period boundaries are locked to January 1, 2023 through March 31, 2026, the congestion\-pricing cutoff is fixed at January 5, 2025, and the notebook is configured to run in conservative pilot mode\. Full weather expansion is still disabled, rebuilds are off by default, and the output paths are already pointed at the existing pipeline\_data/1\.1\.7\.\.\. structure rather than introducing a new storage pattern\.

\-\-\-

## 1\.1\.7\.1 Weather Source and Station Configuration

This section defines the weather\-source strategy before we touch any raw observations\. Rather than starting with dynamic station discovery or a broad regional pull, we keep the first implementation explicit and reviewable: NOAA as the source, a small fixed pilot station list, and a slightly expanded station list held in reserve for later\. This also gives us a clear expected raw schema and a canonical staged schema up front, so we can validate the source configuration before we spend time building the pilot load\.

In [3]:
# ---------------------------------------------------------------------
# 1.1.7.1 NOAA source and station configuration
# ---------------------------------------------------------------------

WEATHER_SOURCE_NAME = "NOAA"
WEATHER_SOURCE_DATASET = "hourly_station_observations"
WEATHER_SOURCE_METHOD = "explicit_station_list"
WEATHER_SOURCE_NOTES = (
    "Use a fixed NOAA station list for 1.1.7. "
    "Do not use dynamic station discovery in the first implementation."
)

# These pilot IDs are explicit starting points for the notebook configuration.
# We can revise them later if the selected NOAA extraction method uses a
# different station identifier format.
PILOT_WEATHER_STATIONS = [
    {
        "weather_station_id": "USW00014732",
        "weather_station_name": "New York LaGuardia Airport",
        "source_station_id": "USW00014732",
        "source_station_name": "New York LaGuardia Airport",
        "source_timezone": PROJECT_TIMEZONE,
    },
    {
        "weather_station_id": "USW00094789",
        "weather_station_name": "New York JFK Airport",
        "source_station_id": "USW00094789",
        "source_station_name": "New York JFK Airport",
        "source_timezone": PROJECT_TIMEZONE,
    },
]

FULL_WEATHER_STATIONS = [
    *PILOT_WEATHER_STATIONS,
    {
        "weather_station_id": "USW00014734",
        "weather_station_name": "Newark Liberty International Airport",
        "source_station_id": "USW00014734",
        "source_station_name": "Newark Liberty International Airport",
        "source_timezone": PROJECT_TIMEZONE,
    },
    {
        "weather_station_id": "USW00094728",
        "weather_station_name": "New York Central Park",
        "source_station_id": "USW00094728",
        "source_station_name": "NY CITY CENTRAL PARK, NY US",
        "source_timezone": PROJECT_TIMEZONE,
    },
]

MINIMUM_REQUIRED_RAW_WEATHER_COLUMNS = [
    "station_id",
    "station_name",
    "observation_datetime",
    "temperature_f",
]

OPTIONAL_RAW_WEATHER_COLUMNS = [
    "precipitation_in",
    "snowfall_in",
    "wind_speed_mph",
    "wind_gust_mph",
    "visibility_miles",
    "weather_condition",
    "latitude",
    "longitude",
]

CANONICAL_WEATHER_COLUMNS = [
    "weather_station_id",
    "weather_station_name",
    "observation_datetime",
    "date",
    "year",
    "month",
    "day_of_week",
    "hour",
    "pre_post_cp",
    "temperature_f",
    "precipitation_in",
    "snowfall_in",
    "wind_speed_mph",
    "wind_gust_mph",
    "visibility_miles",
    "weather_condition",
    "source_dataset",
    "source_station_id",
    "source_station_name",
    "source_latitude",
    "source_longitude",
    "source_timezone",
    "weather_observation_method",
    "weather_observation_source",
    "weather_spatial_role",
    "canonical_geography",
]


def validate_station_list(stations, label: str) -> None:
    """Check that station configuration is explicit before loading data."""
    required_keys = {
        "weather_station_id",
        "weather_station_name",
        "source_station_id",
        "source_station_name",
        "source_timezone",
    }

    if not stations:
        raise AssertionError(f"{label} station list is empty.")

    seen_station_ids = set()

    for station in stations:
        missing_keys = sorted(required_keys - set(station))
        if missing_keys:
            raise AssertionError(
                f"{label} station is missing required keys: {missing_keys}"
            )

        station_id = station["weather_station_id"]
        if station_id in seen_station_ids:
            raise AssertionError(
                f"Duplicate weather_station_id found in {label}: {station_id}"
            )

        seen_station_ids.add(station_id)


validate_station_list(PILOT_WEATHER_STATIONS, "pilot")
validate_station_list(FULL_WEATHER_STATIONS, "full")

pilot_station_ids = {
    station["weather_station_id"]
    for station in PILOT_WEATHER_STATIONS
}
full_station_ids = {
    station["weather_station_id"]
    for station in FULL_WEATHER_STATIONS
}

if not pilot_station_ids.issubset(full_station_ids):
    raise AssertionError(
        "Full weather station list must contain all pilot weather stations."
    )

weather_source_summary_df = pd.DataFrame(
    {
        "setting": [
            "weather_source_name",
            "weather_source_dataset",
            "weather_source_method",
            "pilot_station_count",
            "full_station_count",
            "minimum_required_raw_weather_columns",
            "optional_raw_weather_columns",
            "canonical_weather_columns",
        ],
        "value": [
            WEATHER_SOURCE_NAME,
            WEATHER_SOURCE_DATASET,
            WEATHER_SOURCE_METHOD,
            len(PILOT_WEATHER_STATIONS),
            len(FULL_WEATHER_STATIONS),
            ", ".join(MINIMUM_REQUIRED_RAW_WEATHER_COLUMNS),
            ", ".join(OPTIONAL_RAW_WEATHER_COLUMNS),
            ", ".join(CANONICAL_WEATHER_COLUMNS),
        ],
    }
)

pilot_weather_station_config_df = pd.DataFrame(PILOT_WEATHER_STATIONS)
full_weather_station_config_df = pd.DataFrame(FULL_WEATHER_STATIONS)

print_step("NOAA weather source configuration loaded.")
display(weather_source_summary_df)
display(pilot_weather_station_config_df)
display(full_weather_station_config_df)

[1.1.7] NOAA weather source configuration loaded.


,setting,value
0,weather_source_name,NOAA
1,weather_source_dataset,hourly_station_observations
2,weather_source_method,explicit_station_list
3,pilot_station_count,2
4,full_station_count,4
5,minimum_required_raw_weather_columns,"station_id, station_name, observation_datetime, temperature_f"
6,optional_raw_weather_columns,"precipitation_in, snowfall_in, wind_speed_mph, wind_gust_mph, visibility_miles, weather_condition, latitude, longitude"
7,canonical_weather_columns,"weather_station_id, weather_station_name, observation_datetime, date, year, month, day_of_week, hour, pre_post_cp, temperature_f, precipitation_in, snowfall_in, wind_speed_mph, wind_gust_mph, visibility_miles, weather_condition, source_dataset, source_station_id, source_station_name, source_latitude, source_longitude, source_timezone, weather_observation_method, weather_observation_source, weather_spatial_role, canonical_geography"


,weather_station_id,weather_station_name,source_station_id,source_station_name,source_timezone
0,USW00014732,New York LaGuardia Airport,USW00014732,New York LaGuardia Airport,America/New_York
1,USW00094789,New York JFK Airport,USW00094789,New York JFK Airport,America/New_York


,weather_station_id,weather_station_name,source_station_id,source_station_name,source_timezone
0,USW00014732,New York LaGuardia Airport,USW00014732,New York LaGuardia Airport,America/New_York
1,USW00094789,New York JFK Airport,USW00094789,New York JFK Airport,America/New_York
2,USW00014734,Newark Liberty International Airport,USW00014734,Newark Liberty International Airport,America/New_York
3,USW00094728,New York Central Park,USW00094728,"NY CITY CENTRAL PARK, NY US",America/New_York


Findings\. The source configuration is now clean and aligned with the station universe we actually carried through the notebook\. We are still using NOAA hourly station observations with an explicit station\-list strategy, but the canonical full list now includes the four stations that matter for this workflow: LaGuardia, JFK, Newark, and Central Park\. That is the right fix here, because it keeps downstream station\-name enrichment and manifests tied to the same explicit configuration rather than letting pilot\-discovered stations drift outside the canonical reference list\.

\-\-\-

## 1\.1\.7\.2 Pilot load: Canonical weather stations

This section uses one canonical station\-year file to verify that the NOAA hourly weather source is actually usable before scaling up to the full station set\. The goal is to answer the key feasibility questions: can we retrieve the source cleanly, do the timestamps parse, does the file cover the expected year, are duplicate within\-hour observations manageable, and are the core weather fields present often enough to support a practical staged hourly table\.

Operationally, this section should stay narrow and exploratory\. It should focus on one pilot station, inspect the raw schema, quantify timestamp behavior and within\-hour duplication, and evaluate the coverage of the core weather variables we care about for mobility analysis\. By the end of the section, we should know whether the hourly source is good enough to proceed and what cleanup rules the later staging logic will need\.

### NOAA station discovery

This block defines the NOAA API session and the geographic search window for NYC\-area weather station discovery\. The goal is to keep configuration separate from discovery logic so the station search can be rerun or adjusted without rewriting downstream code\.

In [4]:
# --------------------------------------------------
# 1.1.7.2.1a Define NOAA station-discovery configuration
# --------------------------------------------------

NOAA_API_TOKEN = os.environ.get("NOAA_API_TOKEN")
if not NOAA_API_TOKEN:
    raise RuntimeError("Missing NOAA_API_TOKEN in Deepnote environment variables.")

BASE_API = "https://www.ncei.noaa.gov/cdo-web/api/v2"
HEADERS = {"token": NOAA_API_TOKEN}

SESSION = requests.Session()
SESSION.headers.update(HEADERS)

# Rough NYC-region bounding box: south, west, north, east
NYC_EXTENT = "40.45,-74.30,41.05,-73.65"

This block defines the minimal NOAA API helper functions used in the notebook\. These helpers should only handle authenticated GET requests and pagination\. They should not contain any project\-specific station filtering or weather\-staging logic\.

In [5]:
# --------------------------------------------------
# 1.1.7.2.1b Define NOAA API helpers
# --------------------------------------------------

def cdo_get(endpoint: str, params: Optional[Dict] = None) -> Dict:
    url = f"{BASE_API}/{endpoint}"
    resp = SESSION.get(url, params=params, timeout=60)
    resp.raise_for_status()
    return resp.json()


def paginate_cdo(endpoint: str, params: Optional[Dict] = None, sleep_s: float = 0.25) -> List[Dict]:
    params = dict(params or {})
    params.setdefault("limit", 1000)

    all_rows: List[Dict] = []
    offset = 1

    while True:
        params["offset"] = offset
        payload = cdo_get(endpoint, params=params)
        rows = payload.get("results", [])
        all_rows.extend(rows)

        meta = payload.get("metadata", {}).get("resultset", {})
        count = meta.get("count", 0)
        limit = meta.get("limit", params["limit"])

        if offset + limit > count:
            break

        offset += limit
        time.sleep(sleep_s)

    return all_rows

This block queries NOAA for stations within the NYC\-region bounding box and creates a first\-pass candidate station table\. The goal is to identify the canonical airport and major\-station records that could support the hourly weather workflow, not to finalize the production station list yet\.

In [6]:
# --------------------------------------------------
# 1.1.7.2.1c Discover candidate NYC-area NOAA stations
# --------------------------------------------------

stations = paginate_cdo(
    "stations",
    params={
        "extent": NYC_EXTENT,
        "startdate": "2023-01-01",
        "enddate": "2026-03-31",
        "sortfield": "name",
        "sortorder": "asc",
        "limit": 1000,
    },
)

candidate_weather_stations_df = (
    pd.DataFrame(stations)
    .sort_values("name")
    .reset_index(drop=True)
)

print(f"Found {len(candidate_weather_stations_df)} NOAA stations in the NYC bounding box.")
display(candidate_weather_stations_df[["id", "name", "latitude", "longitude", "mindate", "maxdate", "datacoverage"]])

Found 80 NOAA stations in the NYC bounding box.


,id,name,latitude,longitude,mindate,maxdate,datacoverage
0,GHCND:US1NYNS0067,"ALBERTSON 0.5 SW, NY US",40.765735,-73.655448,2021-07-08,2023-09-22,0.2689
1,GHCND:US1NYKN0059,"BROOKLYN 2.4 WSW, NY US",40.659680,-73.982799,2022-08-07,2026-07-09,0.4892
2,GHCND:US1NYKN0025,"BROOKLYN 3.1 NW, NY US",40.684600,-73.986700,2013-11-29,2024-07-16,0.8241
3,WBAN:54743,"CALDWELL ESSEX CO AIRPORT, NJ US",40.876450,-74.282840,2005-01-01,2025-08-27,1.0000
4,GHCND:USW00054743,"CALDWELL ESSEX CO AIRPORT, NJ US",40.876450,-74.282840,1999-06-01,2026-07-10,0.9969
5,GHCND:US1NJES0024,"CEDAR GROVE TWP 0.4 W, NJ US",40.855695,-74.235564,1998-06-17,2026-04-30,0.9922
6,GHCND:US1NJUN0036,"CLARK TWP 1.3 ENE, NJ US",40.626055,-74.289674,2020-04-28,2026-07-11,1.0000
7,GHCND:US1NJUN0035,"CRANFORD TWP 1.1 ESE, NJ US",40.652357,-74.284536,2021-07-30,2026-07-11,0.6952
8,GHCND:US1NYNS0077,"EAST ROCKAWAY 0.5 S, NY US",40.636770,-73.667170,2023-04-12,2024-05-18,0.9925
9,GHCND:US1NYWC0019,"ELMSFORD 0.8 SSW, NY US",41.042666,-73.822799,2017-06-30,2023-12-18,0.2031


Findings\. The initial NOAA station\-discovery pull confirms that the NYC\-region search window is broad enough to recover a large and mixed set of candidate weather stations\. The output includes neighborhood\-scale stations, airport stations, and multiple station\-ID systems for the same physical location, which is useful because it means the search is not missing major anchors like JFK, LaGuardia, and Central Park, but it also makes clear that the raw result set is too noisy to use directly\. In practice, this step did what it needed to do: it proved that NOAA station discovery works for the study area and that the next step should be narrowing the station universe to the major canonical stations that are most likely to have stable hourly files and long coverage across the full study window\.

This block filters the broader NOAA station table down to the most plausible major\-station candidates for the GHCNh hourly workflow\. The goal is to isolate the airport and canonical stations that are most likely to have long, stable, machine\-readable station\-year files\.

In [7]:
# --------------------------------------------------
# 1.1.7.2.1d Narrow to likely major hourly-weather stations
# --------------------------------------------------

likely_major_weather_stations_df = (
    candidate_weather_stations_df[
        candidate_weather_stations_df["id"].str.startswith(("WBAN:", "GHCND:USW"), na=False)
    ]
    .copy()
    .sort_values("name")
    .reset_index(drop=True)
)

display(
    likely_major_weather_stations_df[
        ["id", "name", "latitude", "longitude", "mindate", "maxdate", "datacoverage"]
    ]
)

,id,name,latitude,longitude,mindate,maxdate,datacoverage
0,WBAN:54743,"CALDWELL ESSEX CO AIRPORT, NJ US",40.87645,-74.28284,2005-01-01,2025-08-27,1.0000
1,GHCND:USW00054743,"CALDWELL ESSEX CO AIRPORT, NJ US",40.87645,-74.28284,1999-06-01,2026-07-10,0.9969
2,WBAN:94789,"JFK INTERNATIONAL AIRPORT, NY US",40.63915,-73.76390,1948-07-01,2025-08-27,1.0000
3,GHCND:USW00094789,"JFK INTERNATIONAL AIRPORT, NY US",40.63915,-73.76390,1948-01-01,2026-07-10,0.9999
4,WBAN:14732,"LAGUARDIA AIRPORT, NY US",40.77945,-73.88027,1948-07-01,2025-08-27,1.0000
5,GHCND:USW00014732,"LAGUARDIA AIRPORT, NY US",40.77945,-73.88027,1939-10-07,2026-07-10,1.0000
6,WBAN:00178,"LINDEN AIRPORT, NJ US",40.61700,-74.25000,2014-07-31,2025-08-25,1.0000
7,WBAN:14734,"NEWARK LIBERTY INTERNATIONAL AIRPORT, NJ US",40.68275,-74.16927,1948-01-01,2025-08-25,1.0000
8,GHCND:USW00014734,"NEWARK LIBERTY INTERNATIONAL AIRPORT, NJ US",40.68275,-74.16927,1843-01-01,2026-07-10,0.9486
9,WBAN:94728,"NY CITY CENTRAL PARK, NY US",40.77898,-73.96925,1943-12-01,2025-08-27,0.7100


Findings\. Filtering the broader NOAA station search down to WBAN: and GHCND:USW records gives us a much cleaner set of major candidate stations for the hourly weather workflow\. The expected anchors are all there — Central Park, JFK, LaGuardia, Newark, Teterboro, Caldwell, and Linden — and most show long coverage windows and near\-complete data coverage, which is a good sign that the station universe is now narrow enough to build a practical pilot manifest from\.

### Define GHCNh File Access Pattern

This block derives the station codes needed for GHCNh station\-year file downloads\. The goal is to convert the NOAA station identifiers into the station\-code format used by the hourly access files, while preserving station metadata for later manifest construction\.

In [8]:
# --------------------------------------------------
# 1.1.7.2.2a Derive GHCNh station codes
# --------------------------------------------------
def derive_ghcnh_station_code(station_id: str) -> Optional[str]:
    if isinstance(station_id, str) and station_id.startswith("GHCND:USW"):
        return station_id.split(":", 1)[1]
    if isinstance(station_id, str) and station_id.startswith("WBAN:"):
        suffix = station_id.split(":", 1)[1]
        if suffix.isdigit() and len(suffix) == 5:
            return f"USW000{suffix}"
    return None


likely_major_weather_stations_df["station_code"] = likely_major_weather_stations_df["id"].apply(derive_ghcnh_station_code)

ghcnh_candidate_stations_df = (
    likely_major_weather_stations_df[
        likely_major_weather_stations_df["station_code"].notna()
    ]
    .copy()
    .sort_values(["name", "station_code"])
    .reset_index(drop=True)
)

display(
    ghcnh_candidate_stations_df[
        ["id", "station_code", "name", "latitude", "longitude", "mindate", "maxdate", "datacoverage"]
    ]
)

,id,station_code,name,latitude,longitude,mindate,maxdate,datacoverage
0,WBAN:54743,USW00054743,"CALDWELL ESSEX CO AIRPORT, NJ US",40.87645,-74.28284,2005-01-01,2025-08-27,1.0000
1,GHCND:USW00054743,USW00054743,"CALDWELL ESSEX CO AIRPORT, NJ US",40.87645,-74.28284,1999-06-01,2026-07-10,0.9969
2,WBAN:94789,USW00094789,"JFK INTERNATIONAL AIRPORT, NY US",40.63915,-73.76390,1948-07-01,2025-08-27,1.0000
3,GHCND:USW00094789,USW00094789,"JFK INTERNATIONAL AIRPORT, NY US",40.63915,-73.76390,1948-01-01,2026-07-10,0.9999
4,WBAN:14732,USW00014732,"LAGUARDIA AIRPORT, NY US",40.77945,-73.88027,1948-07-01,2025-08-27,1.0000
5,GHCND:USW00014732,USW00014732,"LAGUARDIA AIRPORT, NY US",40.77945,-73.88027,1939-10-07,2026-07-10,1.0000
6,WBAN:00178,USW00000178,"LINDEN AIRPORT, NJ US",40.61700,-74.25000,2014-07-31,2025-08-25,1.0000
7,WBAN:14734,USW00014734,"NEWARK LIBERTY INTERNATIONAL AIRPORT, NJ US",40.68275,-74.16927,1948-01-01,2025-08-25,1.0000
8,GHCND:USW00014734,USW00014734,"NEWARK LIBERTY INTERNATIONAL AIRPORT, NJ US",40.68275,-74.16927,1843-01-01,2026-07-10,0.9486
9,WBAN:94728,USW00094728,"NY CITY CENTRAL PARK, NY US",40.77898,-73.96925,1943-12-01,2025-08-27,0.7100


Findings\. Deriving the GHCNh station codes worked cleanly and gave us a consistent station\-code field we can use to construct station\-year file paths\. The main thing this surfaced is that some stations appear twice — once as WBAN: and once as GHCND:USW\.\.\. — but both resolve to the same hourly station code, which means the next step should be deduplicating at the station\_code level before building the download manifest\.

This block defines the deterministic NOAA GHCNh station\-year URL pattern and the local target\-path pattern\. The goal is to separate endpoint construction from download execution so the notebook can inspect the full set of required files before pulling any data\.

In [9]:
# --------------------------------------------------
# 1.1.7.2.2b Define GHCNh station-year endpoint pattern
# --------------------------------------------------

GHCNH_BASE = "https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year"
RAW_GHCNH_DIR = Path("source_data/weather/noaa/ghcnh/station_year")


def ghcnh_station_year_url(station_code: str, year: int) -> str:
    return f"{GHCNH_BASE}/{year}/psv/GHCNh_{station_code}_{year}.psv"


def ghcnh_station_year_target_path(station_code: str, year: int, out_dir: Path = RAW_GHCNH_DIR) -> Path:
    return out_dir / f"GHCNh_{station_code}_{year}.psv"

This block builds the station\-year manifest for the full study period\. The goal is to list every hourly GHCNh endpoint we expect to need, along with the local target path where each downloaded file should be stored\.

In [10]:
# --------------------------------------------------
# 1.1.7.2.2b Build the candidate station-year manifest
# --------------------------------------------------

study_years = [2023, 2024, 2025, 2026]

manifest_rows = []

for _, row in ghcnh_candidate_stations_df.iterrows():
    for year in study_years:
        manifest_rows.append(
            {
                "station_id": row["id"],
                "station_code": row["station_code"],
                "station_name": row["name"],
                "year": year,
                "ghcnh_url": ghcnh_station_year_url(row["station_code"], year),
                "target_path": str(ghcnh_station_year_target_path(row["station_code"], year)),
            }
        )

station_year_manifest_df = pd.DataFrame(manifest_rows).sort_values(
    ["station_name", "year"]
).reset_index(drop=True)

display(station_year_manifest_df)

,station_id,station_code,station_name,year,ghcnh_url,target_path
0,WBAN:54743,USW00054743,"CALDWELL ESSEX CO AIRPORT, NJ US",2023,https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2023/psv/GHCNh_USW00054743_2023.psv,source_data/weather/noaa/ghcnh/station_year/GHCNh_USW00054743_2023.psv
1,GHCND:USW00054743,USW00054743,"CALDWELL ESSEX CO AIRPORT, NJ US",2023,https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2023/psv/GHCNh_USW00054743_2023.psv,source_data/weather/noaa/ghcnh/station_year/GHCNh_USW00054743_2023.psv
2,WBAN:54743,USW00054743,"CALDWELL ESSEX CO AIRPORT, NJ US",2024,https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2024/psv/GHCNh_USW00054743_2024.psv,source_data/weather/noaa/ghcnh/station_year/GHCNh_USW00054743_2024.psv
3,GHCND:USW00054743,USW00054743,"CALDWELL ESSEX CO AIRPORT, NJ US",2024,https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2024/psv/GHCNh_USW00054743_2024.psv,source_data/weather/noaa/ghcnh/station_year/GHCNh_USW00054743_2024.psv
4,WBAN:54743,USW00054743,"CALDWELL ESSEX CO AIRPORT, NJ US",2025,https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2025/psv/GHCNh_USW00054743_2025.psv,source_data/weather/noaa/ghcnh/station_year/GHCNh_USW00054743_2025.psv
5,GHCND:USW00054743,USW00054743,"CALDWELL ESSEX CO AIRPORT, NJ US",2025,https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2025/psv/GHCNh_USW00054743_2025.psv,source_data/weather/noaa/ghcnh/station_year/GHCNh_USW00054743_2025.psv
6,WBAN:54743,USW00054743,"CALDWELL ESSEX CO AIRPORT, NJ US",2026,https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2026/psv/GHCNh_USW00054743_2026.psv,source_data/weather/noaa/ghcnh/station_year/GHCNh_USW00054743_2026.psv
7,GHCND:USW00054743,USW00054743,"CALDWELL ESSEX CO AIRPORT, NJ US",2026,https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2026/psv/GHCNh_USW00054743_2026.psv,source_data/weather/noaa/ghcnh/station_year/GHCNh_USW00054743_2026.psv
8,WBAN:94789,USW00094789,"JFK INTERNATIONAL AIRPORT, NY US",2023,https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2023/psv/GHCNh_USW00094789_2023.psv,source_data/weather/noaa/ghcnh/station_year/GHCNh_USW00094789_2023.psv
9,GHCND:USW00094789,USW00094789,"JFK INTERNATIONAL AIRPORT, NY US",2023,https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2023/psv/GHCNh_USW00094789_2023.psv,source_data/weather/noaa/ghcnh/station_year/GHCNh_USW00094789_2023.psv


Findings\. The manifest build worked and gave us a deterministic station\-year download plan for the full study period, with explicit NOAA endpoints and local target paths for each station\-year file\. The main issue it surfaced is duplication: because both WBAN: and GHCND:USW\.\.\. records were carried forward, the manifest currently includes repeated URLs and target paths for the same underlying station code\. So the structure is now right, but the next cleanup step is to deduplicate the manifest at the station\_code × year level before doing any bulk downloads\.

### Build the candidate station\-year manifest

This block creates a small pilot manifest that can be used to test the hourly download flow on one or two canonical stations before scaling up to the full weather station set\.

In [11]:
# --------------------------------------------------
# 1.1.7.2.3a Create the pilot station-year manifest
# --------------------------------------------------

pilot_station_codes = ["USW00094728"]  # Central Park first

pilot_station_year_manifest_df = (
    station_year_manifest_df[
        station_year_manifest_df["station_code"].isin(pilot_station_codes)
        & (station_year_manifest_df["year"] == 2023)
    ]
    .copy()
    .reset_index(drop=True)
)

display(pilot_station_year_manifest_df)

,station_id,station_code,station_name,year,ghcnh_url,target_path
0,WBAN:94728,USW00094728,"NY CITY CENTRAL PARK, NY US",2023,https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2023/psv/GHCNh_USW00094728_2023.psv,source_data/weather/noaa/ghcnh/station_year/GHCNh_USW00094728_2023.psv
1,GHCND:USW00094728,USW00094728,"NY CITY CENTRAL PARK, NY US",2023,https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2023/psv/GHCNh_USW00094728_2023.psv,source_data/weather/noaa/ghcnh/station_year/GHCNh_USW00094728_2023.psv


Findings\. The pilot manifest did what it was supposed to do: it narrowed the full station\-year plan down to a single Central Park 2023 endpoint so we could test the hourly download flow on one canonical station before scaling up\. It also makes the remaining duplication issue obvious, since both the WBAN: and GHCND:USW\.\.\. records collapse to the exact same station code, URL, and target path, which confirms that future pilot and full manifests should be deduplicated before download\.

This block prints the exact pilot and full\-study endpoints implied by the current station manifest\. The goal is to make the download plan explicit and inspectable before any file retrieval occurs\.

In [12]:
# --------------------------------------------------
# 1.1.7.2.3b Preview the endpoints we expect to download
# --------------------------------------------------

print("Pilot endpoints:")
for _, row in pilot_station_year_manifest_df.iterrows():
    print(row["ghcnh_url"])

print("\nFirst 20 full-manifest endpoints:")
for url in station_year_manifest_df["ghcnh_url"].head(20):
    print(url)

Pilot endpoints:
https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2023/psv/GHCNh_USW00094728_2023.psv
https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2023/psv/GHCNh_USW00094728_2023.psv

First 20 full-manifest endpoints:
https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2023/psv/GHCNh_USW00054743_2023.psv
https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2023/psv/GHCNh_USW00054743_2023.psv
https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2024/psv/GHCNh_USW00054743_2024.psv
https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2024/psv/GHCNh_USW00054743_2024.psv
https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2025/psv/GHCNh_USW00054743_2025.psv
https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly

Findings: Previewing the pilot and full\-manifest endpoints confirms that the GHCNh URL pattern is stable and easy to generate once the station code and year are known\. It also confirms again that the current manifest still carries duplicate endpoints because both WBAN: and GHCND:USW\.\.\. station records resolve to the same station\-year files, so the endpoint logic itself is fine — the cleanup we still need is deduplication before any multi\-file download step\.

### Create and Inspect Pilot Download Plan

This block defines the minimal helper used to download one hourly GHCNh station\-year file from NOAA’s HTTPS access path\. The GHCNh documentation explicitly supports station\-year downloads and gives the exact file pattern used by the access folders\. This helper should only build the URL, download the file, and save it locally\. It should not standardize, aggregate, or merge the weather data yet\.

In [13]:
# --------------------------------------------------
# 1.1.7.2.4a GHCNh hourly download helper
# --------------------------------------------------

GHCNH_BASE = "https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year"
RAW_GHCNH_DIR = Path("source_data/weather/noaa/ghcnh/station_year")


def ghcnh_station_year_url(station_code: str, year: int) -> str:
    return f"{GHCNH_BASE}/{year}/psv/GHCNh_{station_code}_{year}.psv"


def ghcnh_station_year_target_path(station_code: str, year: int, out_dir: Path = RAW_GHCNH_DIR) -> Path:
    return out_dir / f"GHCNh_{station_code}_{year}.psv"


def download_ghcnh_station_year(station_code: str, year: int, out_dir: Path = RAW_GHCNH_DIR) -> Path:
    out_dir.mkdir(parents=True, exist_ok=True)

    url = ghcnh_station_year_url(station_code, year)
    out_path = ghcnh_station_year_target_path(station_code, year, out_dir=out_dir)

    print(f"Downloading: {url}")
    resp = requests.get(url, timeout=120)
    resp.raise_for_status()

    out_path.write_bytes(resp.content)

    print(f"Saved to: {out_path}")
    print(f"File size: {out_path.stat().st_size:,} bytes")
    return out_path

This block performs the first real hourly weather download for the notebook using Central Park as the canonical pilot station\. The goal is simply to prove that the correct hourly NOAA dataset can be pulled successfully for the right station and saved into the raw weather staging folder\. 

In [14]:
# --------------------------------------------------
# 1.1.7.2.4b Download one canonical hourly station-year file
# --------------------------------------------------

pilot_station_year_manifest_dedup_df = (
    pilot_station_year_manifest_df
    .drop_duplicates(subset=["station_code", "year"])
    .reset_index(drop=True)
)

if pilot_station_year_manifest_dedup_df.empty:
    raise RuntimeError("Pilot station-year manifest is empty.")

pilot_row = pilot_station_year_manifest_dedup_df.iloc[0]

station_code = pilot_row["station_code"]
year = int(pilot_row["year"])

ghcnh_path = download_ghcnh_station_year(
    station_code=station_code,
    year=year,
    out_dir=RAW_GHCNH_DIR,
)

Downloading: https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2023/psv/GHCNh_USW00094728_2023.psv
Saved to: source_data/weather/noaa/ghcnh/station_year/GHCNh_USW00094728_2023.psv
File size: 10,804,486 bytes


This block performs a lightweight inspection of the downloaded GHCNh station\-year file so we can confirm that it is hourly data, not daily data, and that the schema is usable for downstream staging\. The goal here is only to inspect the raw file shape and sample records before standardization\.

In [15]:
# --------------------------------------------------
# 1.1.7.2.4c Inspect the downloaded hourly file
# --------------------------------------------------

ghcnh_sample = pd.read_csv(ghcnh_path, sep="|", nrows=20)

print("Downloaded file:")
print(ghcnh_path)

print("\nColumns:")
print(list(ghcnh_sample.columns))

display(ghcnh_sample)

Downloaded file:
source_data/weather/noaa/ghcnh/station_year/GHCNh_USW00094728_2023.psv

Columns:
['STATION', 'Station_name', 'DATE', 'Year', 'Month', 'Day', 'Hour', 'Minute', 'LATITUDE', 'LONGITUDE', 'ELEVATION', 'temperature', 'temperature_Measurement_Code', 'temperature_Quality_Code', 'temperature_Report_Type', 'temperature_Source_Code', 'temperature_Source_Station_ID', 'dew_point_temperature', 'dew_point_temperature_Measurement_Code', 'dew_point_temperature_Quality_Code', 'dew_point_temperature_Report_Type', 'dew_point_temperature_Source_Code', 'dew_point_temperature_Source_Station_ID', 'station_level_pressure', 'station_level_pressure_Measurement_Code', 'station_level_pressure_Quality_Code', 'station_level_pressure_Report_Type', 'station_level_pressure_Source_Code', 'station_level_pressure_Source_Station_ID', 'sea_level_pressure', 'sea_level_pressure_Measurement_Code', 'sea_level_pressure_Quality_Code', 'sea_level_pressure_Report_Type', 'sea_level_pressure_Source_Code', 'sea_level

,STATION,Station_name,DATE,Year,Month,Day,Hour,Minute,LATITUDE,LONGITUDE,...,precipitation_24_hour_Quality_Code,precipitation_24_hour_Report_Type,precipitation_24_hour_Source_Code,precipitation_24_hour_Source_Station_ID,REM,REM_Measurement_Code,REM_Quality_Code,REM_Report_Type,REM_Source_Code,REM_Source_Station_ID
0,USW00094728,NY CITY CNTRL PARK,2023-01-01T00:06:00,2023,1,1,0,6,40.7789,-73.9692,...,NaN,NaN,NaN,NaN,MET11812/31/22 19:06:03 SPECI KNYC 010006Z 00000KT 1 1/2SM -RA BR OVC005 10/09 A2986 RMK AO2 CIG 004V008 P0000 T01000089 RTX,NaN,NaN,FM16,343,725053-94728
1,USW00094728,NY CITY CNTRL PARK,2023-01-01T00:18:00,2023,1,1,0,18,40.7789,-73.9692,...,NaN,NaN,NaN,NaN,MET10612/31/22 19:18:03 SPECI KNYC 010018Z 03003KT 2 1/2SM -RA BR OVC006 10/09 A2986 RMK AO2 P0000 T01000094 RTX,NaN,NaN,FM16,343,725053-94728
2,USW00094728,NY CITY CNTRL PARK,2023-01-01T00:31:00,2023,1,1,0,31,40.7789,-73.9692,...,NaN,NaN,NaN,NaN,MET11112/31/22 19:31:03 SPECI KNYC 010031Z 00000KT 1 1/2SM -RA BR OVC006 10/09 A2986 RMK AO2 P0001 T01000089 RTX FIBI,NaN,NaN,FM16,343,725053-94728
3,USW00094728,NY CITY CNTRL PARK,2023-01-01T00:44:00,2023,1,1,0,44,40.7789,-73.9692,...,NaN,NaN,NaN,NaN,MET10212/31/22 19:44:03 SPECI KNYC 010044Z 00000KT 2SM -RA BR OVC006 10/09 A2985 RMK AO2 P0002 T01000094 RTX,NaN,NaN,FM16,343,725053-94728
4,USW00094728,NY CITY CNTRL PARK,2023-01-01T00:49:00,2023,1,1,0,49,40.7789,-73.9692,...,NaN,NaN,NaN,NaN,MET09712/31/22 19:49:03 SPECI KNYC 010049Z 00000KT 1 3/4SM -RA BR OVC007 10/09 A2985 RMK AO2 P0002 FIBI,NaN,NaN,FM16,343,725053-94728
5,USW00094728,NY CITY CNTRL PARK,2023-01-01T00:51:00,2023,1,1,0,51,40.7789,-73.9692,...,NaN,NaN,NaN,NaN,MET12112/31/22 19:51:03 METAR KNYC 010051Z 00000KT 1 3/4SM -RA BR OVC007 10/09 A2985 RMK AO2 CIG 006V010 SLP098 P0002 T01000094,NaN,NaN,FM15,343,725053-94728
6,USW00094728,NY CITY CNTRL PARK,2023-01-01T01:19:00,2023,1,1,1,19,40.7789,-73.9692,...,NaN,NaN,NaN,NaN,MET11812/31/22 20:19:03 SPECI KNYC 010119Z 00000KT 2 1/2SM -RA BR OVC007 11/09 A2983 RMK AO2 CIG 005V010 P0002 T01060094 RTX,NaN,NaN,FM16,343,725053-94728
7,USW00094728,NY CITY CNTRL PARK,2023-01-01T01:27:00,2023,1,1,1,27,40.7789,-73.9692,...,NaN,NaN,NaN,NaN,MET11412/31/22 20:27:03 SPECI KNYC 010127Z VRB03KT 3SM -RA BR OVC005 12/11 A2982 RMK AO2 CIG 004V009 P0002 T01170106 RTX,NaN,NaN,FM16,343,725053-94728
8,USW00094728,NY CITY CNTRL PARK,2023-01-01T01:51:00,2023,1,1,1,51,40.7789,-73.9692,...,NaN,NaN,NaN,NaN,MET11412/31/22 20:51:03 METAR KNYC 010151Z 25006KT 4SM -RA BR OVC005 13/12 A2981 RMK AO2 SLP086 P0004 T01280117 RTX FIBI,NaN,NaN,FM15,343,725053-94728
9,USW00094728,NY CITY CNTRL PARK,2023-01-01T02:49:00,2023,1,1,2,49,40.7789,-73.9692,...,NaN,NaN,NaN,NaN,MET09612/31/22 21:49:03 SPECI KNYC 010249Z VRB05KT 2 1/2SM RA BR OVC005 12/12 A2981 RMK AO2 P0010 FIBI,NaN,NaN,FM16,343,725053-94728


Findings\. The Central Park station\-year output confirms that the GHCNh file gives us the granular weather data we were looking for\. Records are timestamped down to the minute rather than just the day, and the schema includes a broad set of hourly weather measures such as temperature, dew point, wind speed and direction, precipitation, relative humidity, visibility, pressure, sky conditions, and present\-weather codes, along with extensive measurement, quality, report\-type, and source metadata\. In other words, the raw file is rich enough to support a real hourly weather staging workflow, but it is also much wider and messier than the mobility tables, so the next step will be to decide which timestamp to treat as canonical within each hour, which weather variables are worth keeping, and how much of the quality/source metadata should survive into the staged output\.

### Preview the downloaded pilot

Let's check the observed timestamp span of the downloaded pilot file and confirms whether the station\-year file appears to cover the expected calendar year\. The goal is to verify that timestamps parse cleanly and that the pilot file is aligned with the requested station\-year before we move on to hourly QA\.

In [16]:
# --------------------------------------------------
# 1.1.7.2.5 Check pilot file date range and timestamp coverage
# --------------------------------------------------

ghcnh_full_df = pd.read_csv(ghcnh_path, sep="|")
ghcnh_full_df["DATE"] = pd.to_datetime(ghcnh_full_df["DATE"], errors="coerce")

coverage_summary = pd.DataFrame(
    {
        "station_code": [ghcnh_full_df["STATION"].iloc[0]],
        "station_name": [ghcnh_full_df["Station_name"].iloc[0] if "Station_name" in ghcnh_full_df.columns else None],
        "row_count": [len(ghcnh_full_df)],
        "min_timestamp": [ghcnh_full_df["DATE"].min()],
        "max_timestamp": [ghcnh_full_df["DATE"].max()],
        "null_timestamp_count": [ghcnh_full_df["DATE"].isna().sum()],
        "unique_dates": [ghcnh_full_df["DATE"].dt.date.nunique()],
    }
)

display(coverage_summary)

/tmp/ipykernel_1319/1880925143.py:5: DtypeWarning: Columns (13,19,43,55,91,133,217,223,241,247) have mixed types. Specify dtype option on import or set low_memory=False.
  ghcnh_full_df = pd.read_csv(ghcnh_path, sep="|")


,station_code,station_name,row_count,min_timestamp,max_timestamp,null_timestamp_count,unique_dates
0,USW00094728,NY CITY CNTRL PARK,11464,2023-01-01 00:06:00,2023-12-31 23:51:00,0,365


Findings\. The pilot Central Park station\-year file appears to cover the full 2023 calendar year cleanly\. Timestamps parsed without issue, there were no null timestamps, the file spans from 2023\-01\-01 00:06 through 2023\-12\-31 23:51, and all 365 dates are represented\. So at the year\-level, coverage looks complete enough to move on to the hourly QA questions\.

This block measures how often the pilot file contains multiple observations within the same station\-hour\. The goal is to determine whether duplicate station\-hour records are rare, common but manageable, or frequent enough that we will need a clear within\-hour selection rule before staging the final hourly weather table\.

In [17]:
# --------------------------------------------------
# 1.1.7.2.6 Quantify duplicate hourly observations
# --------------------------------------------------

ghcnh_full_df["date_hour"] = ghcnh_full_df["DATE"].dt.floor("H")

hourly_counts_df = (
    ghcnh_full_df.groupby(["STATION", "date_hour"])
    .size()
    .reset_index(name="records_in_hour")
)

duplicate_hour_summary = pd.DataFrame(
    {
        "station_code": [ghcnh_full_df["STATION"].iloc[0]],
        "total_station_hours": [len(hourly_counts_df)],
        "hours_with_1_record": [(hourly_counts_df["records_in_hour"] == 1).sum()],
        "hours_with_multiple_records": [(hourly_counts_df["records_in_hour"] > 1).sum()],
        "max_records_in_single_hour": [hourly_counts_df["records_in_hour"].max()],
        "avg_records_per_hour": [hourly_counts_df["records_in_hour"].mean()],
    }
)

display(duplicate_hour_summary)

display(
    hourly_counts_df[hourly_counts_df["records_in_hour"] > 1]
    .sort_values(["records_in_hour", "date_hour"], ascending=[False, True])
    .head(25)
)

,station_code,total_station_hours,hours_with_1_record,hours_with_multiple_records,max_records_in_single_hour,avg_records_per_hour
0,USW00094728,8755,7400,1355,9,1.309423


,STATION,date_hour,records_in_hour
5243,USW00094728,2023-08-07 11:00:00,9
6075,USW00094728,2023-09-11 04:00:00,9
6261,USW00094728,2023-09-18 22:00:00,9
6506,USW00094728,2023-09-29 03:00:00,9
7042,USW00094728,2023-10-21 12:00:00,9
2878,USW00094728,2023-04-30 22:00:00,8
4563,USW00094728,2023-07-10 03:00:00,8
5428,USW00094728,2023-08-15 05:00:00,8
5698,USW00094728,2023-08-26 11:00:00,8
5699,USW00094728,2023-08-26 12:00:00,8


Findings\. Duplicate station\-hour observations are common enough that we will need an explicit within\-hour selection rule before staging the final hourly table\. Central Park has 8,755 distinct station\-hours in 2023, and 1,355 of them contain multiple records rather than a single canonical observation; the average is about 1\.31 records per hour, and the busiest hours contain as many as 9 records\. So the duplicates are definitely manageable, but they are frequent enough that we cannot ignore them or assume the raw file is already one\-record\-per\-hour\.

This block checks whether the pilot file contains the core weather measures we actually care about, including temperature, precipitation, wind, visibility, humidity, pressure, snow, and present\-weather fields\. The goal is to confirm that the raw hourly source is rich enough for a practical staged weather table without getting distracted by the full metadata surface\.

In [18]:
# --------------------------------------------------
# 1.1.7.2.7 Check availability of core weather variables
# --------------------------------------------------

core_weather_columns = [
    "temperature",
    "dew_point_temperature",
    "wind_direction",
    "wind_speed",
    "wind_gust",
    "precipitation",
    "relative_humidity",
    "visibility",
    "station_level_pressure",
    "sea_level_pressure",
    "snow_depth",
    "sky_condition",
    "ceiling_height",
    "altimeter",
    "pres_wx_MW1",
    "pres_wx_AU1",
    "precipitation_3_hour",
    "precipitation_6_hour",
    "precipitation_24_hour",
]

availability_rows = []
for col in core_weather_columns:
    if col in ghcnh_full_df.columns:
        non_null_count = ghcnh_full_df[col].notna().sum()
        availability_rows.append(
            {
                "column": col,
                "present_in_file": True,
                "non_null_count": non_null_count,
                "non_null_pct": non_null_count / len(ghcnh_full_df),
            }
        )
    else:
        availability_rows.append(
            {
                "column": col,
                "present_in_file": False,
                "non_null_count": None,
                "non_null_pct": None,
            }
        )

core_weather_availability_df = pd.DataFrame(availability_rows)
display(core_weather_availability_df)

,column,present_in_file,non_null_count,non_null_pct
0,temperature,True,11464,1.000000
1,dew_point_temperature,True,11464,1.000000
2,wind_direction,True,10901,0.950890
3,wind_speed,True,10901,0.950890
4,wind_gust,True,1519,0.132502
5,precipitation,True,9994,0.871773
6,relative_humidity,True,11464,1.000000
7,visibility,True,11456,0.999302
8,station_level_pressure,True,11414,0.995639
9,sea_level_pressure,True,8738,0.762212


Findings\. The core weather fields are mostly present and, for several of the most important measures, coverage is very strong\. Temperature, dew point, and relative humidity are complete; visibility, ceiling height, altimeter, and station\-level pressure are effectively complete; wind direction and wind speed are available in about 95% of records; precipitation is available in about 87%\. The weaker fields are mostly optional enrichments rather than backbone variables — wind gust is sparse, sea\-level pressure is only moderately populated, sky condition is present but incomplete, present\-weather codes are patchy, and the multi\-hour precipitation fields are very sparse\. Snow depth is completely unavailable in this Central Park 2023 file, so it should not be treated as a required hourly feature\.

This block looks at the minute component of pilot\-file timestamps so we can see how observations are distributed within each hour\. The goal is to understand whether the station tends to report on a regular within\-hour cadence and to support a later decision about how to collapse multiple within\-hour observations into one canonical hourly record\.

In [19]:
# --------------------------------------------------
# 1.1.7.2.8 Inspect within-hour observation timing
# --------------------------------------------------

minute_distribution_df = (
    ghcnh_full_df["DATE"]
    .dt.minute
    .value_counts()
    .sort_index()
    .reset_index()
)
minute_distribution_df.columns = ["minute", "observation_count"]

display(minute_distribution_df)

,minute,observation_count
0,0,60
1,1,58
2,2,35
3,3,38
4,4,45
5,5,43
6,6,46
7,7,32
8,8,43
9,9,47


Findings\. Within\-hour observation timing is messy but not random enough to be a blocker\. Records appear across many different minute values rather than landing on one single clean hourly timestamp, which confirms that the raw GHCNh file is not already aligned to a one\-record\-per\-hour structure\. In practice, that means we will need to impose a canonical within\-hour selection rule during staging, but the timestamp pattern itself does not look so erratic that hourly collapse should be difficult\.

\-\-\-

## 1\.1\.7\.3 Choose final station set and load raw station\-year files

This section moves from pilot feasibility into real raw\-data ingestion\. Its job is to finalize which stations belong in the NYC weather workflow, deduplicate the candidate list at the station\-code level, build the full station\-year manifest for the study period, and load the raw GHCNh files into the project source directory\. The output of this section should still be raw source data, not a cleaned or standardized weather table\.

### Deduplicate and Review Candidate Station Set

We’ll start by collapsing the duplicated WBAN: and GHCND:USW\.\.\. records down to one row per station\_code\. This gives us the clean candidate station set we’ll review before deciding which stations to carry into the raw ingest\.

In [20]:
# --------------------------------------------------
# 1.1.7.3.1 Deduplicate candidate stations at the station-code level
# --------------------------------------------------

station_code_dedup_df = (
    ghcnh_candidate_stations_df
    .sort_values(
        by=["station_code", "datacoverage", "maxdate", "mindate"],
        ascending=[True, False, False, True],
    )
    .drop_duplicates(subset=["station_code"], keep="first")
    .reset_index(drop=True)
)

display(
    station_code_dedup_df[
        ["station_code", "name", "latitude", "longitude", "mindate", "maxdate", "datacoverage"]
    ]
)

,station_code,name,latitude,longitude,mindate,maxdate,datacoverage
0,USW00000178,"LINDEN AIRPORT, NJ US",40.61700,-74.25000,2014-07-31,2025-08-25,1.0000
1,USW00014732,"LAGUARDIA AIRPORT, NY US",40.77945,-73.88027,1939-10-07,2026-07-10,1.0000
2,USW00014734,"NEWARK LIBERTY INTERNATIONAL AIRPORT, NJ US",40.68275,-74.16927,1948-01-01,2025-08-25,1.0000
3,USW00054743,"CALDWELL ESSEX CO AIRPORT, NJ US",40.87645,-74.28284,2005-01-01,2025-08-27,1.0000
4,USW00094728,"NY CITY CENTRAL PARK, NY US",40.77898,-73.96925,1869-01-01,2026-07-10,1.0000
5,USW00094741,"TETERBORO AIRPORT, NJ US",40.85898,-74.05616,1969-08-13,2026-07-10,0.9971
6,USW00094789,"JFK INTERNATIONAL AIRPORT, NY US",40.63915,-73.76390,1948-07-01,2025-08-27,1.0000


Findings\. Deduping got us down to seven clean station codes: Central Park, JFK, LaGuardia, Newark, Teterboro, Caldwell, and Linden\. Coverage looks strong across the board, and all seven are viable from a raw\-ingest standpoint\.

Next we’ll check how many raw NOAA station IDs collapsed into each station\_code\. This makes it easy to see where duplicate station representations existed before deduplication\.

In [21]:
# --------------------------------------------------
# 1.1.7.3.2 Summarize source-ID duplication by station code
# --------------------------------------------------

station_code_source_summary_df = (
    ghcnh_candidate_stations_df
    .groupby("station_code", as_index=False)
    .agg(
        raw_station_id_count=("id", "nunique"),
        raw_station_ids=("id", lambda x: sorted(set(x))),
        station_names=("name", lambda x: sorted(set(x))),
    )
    .sort_values(["raw_station_id_count", "station_code"], ascending=[False, True])
    .reset_index(drop=True)
)

display(station_code_source_summary_df)

,station_code,raw_station_id_count,raw_station_ids,station_names
0,USW00014732,2,"[GHCND:USW00014732, WBAN:14732]","[LAGUARDIA AIRPORT, NY US]"
1,USW00014734,2,"[GHCND:USW00014734, WBAN:14734]","[NEWARK LIBERTY INTERNATIONAL AIRPORT, NJ US]"
2,USW00054743,2,"[GHCND:USW00054743, WBAN:54743]","[CALDWELL ESSEX CO AIRPORT, NJ US]"
3,USW00094728,2,"[GHCND:USW00094728, WBAN:94728]","[NY CITY CENTRAL PARK, NY US]"
4,USW00094741,2,"[GHCND:USW00094741, WBAN:94741]","[TETERBORO AIRPORT, NJ US]"
5,USW00094789,2,"[GHCND:USW00094789, WBAN:94789]","[JFK INTERNATIONAL AIRPORT, NY US]"
6,USW00000178,1,[WBAN:00178],"[LINDEN AIRPORT, NJ US]"


Findings\. Most stations were duplicated because NOAA exposed both a WBAN: record and a GHCND:USW\.\.\. record for the same physical station\. That duplication now looks straightforward and manageable: six stations collapse cleanly from two raw IDs to one station code, while Linden only showed up once\.

We’ll also check whether the deduplicated station list still covers the major NYC\-area anchors we expected to find\. This is the first good human review point before we choose the final station set\.

In [22]:
# --------------------------------------------------
# 1.1.7.3.3 Review major deduplicated station candidates
# --------------------------------------------------

major_station_keywords = [
    "CENTRAL PARK",
    "JFK",
    "LAGUARDIA",
    "NEWARK",
    "TETERBORO",
    "CALDWELL",
    "LINDEN",
]

major_station_review_df = station_code_dedup_df[
    station_code_dedup_df["name"].str.upper().apply(
        lambda name: any(keyword in name for keyword in major_station_keywords)
    )
].copy().reset_index(drop=True)

display(
    major_station_review_df[
        ["station_code", "name", "latitude", "longitude", "mindate", "maxdate", "datacoverage"]
    ]
)

,station_code,name,latitude,longitude,mindate,maxdate,datacoverage
0,USW00000178,"LINDEN AIRPORT, NJ US",40.61700,-74.25000,2014-07-31,2025-08-25,1.0000
1,USW00014732,"LAGUARDIA AIRPORT, NY US",40.77945,-73.88027,1939-10-07,2026-07-10,1.0000
2,USW00014734,"NEWARK LIBERTY INTERNATIONAL AIRPORT, NJ US",40.68275,-74.16927,1948-01-01,2025-08-25,1.0000
3,USW00054743,"CALDWELL ESSEX CO AIRPORT, NJ US",40.87645,-74.28284,2005-01-01,2025-08-27,1.0000
4,USW00094728,"NY CITY CENTRAL PARK, NY US",40.77898,-73.96925,1869-01-01,2026-07-10,1.0000
5,USW00094741,"TETERBORO AIRPORT, NJ US",40.85898,-74.05616,1969-08-13,2026-07-10,0.9971
6,USW00094789,"JFK INTERNATIONAL AIRPORT, NY US",40.63915,-73.76390,1948-07-01,2025-08-27,1.0000


Findings\. The cleaned candidate set looks right\. After deduplication, we are left with seven major stations — Central Park, JFK, LaGuardia, Newark, Teterboro, Caldwell, and Linden — all of which have long coverage windows and near\-complete data coverage\. At this point the station universe is small enough to make a deliberate final\-ingest choice rather than carrying forward every plausible regional station\.

Finally, we’ll print a compact summary of the candidate set after deduplication so we know how many stations we are reviewing before making the final raw\-ingest decision\.

In [23]:
# --------------------------------------------------
# 1.1.7.3.4 Summarize deduplicated candidate station set
# --------------------------------------------------

dedup_summary_df = pd.DataFrame(
    {
        "raw_candidate_rows": [len(ghcnh_candidate_stations_df)],
        "deduplicated_station_codes": [station_code_dedup_df["station_code"].nunique()],
        "major_station_review_rows": [len(major_station_review_df)],
    }
)

display(dedup_summary_df)

,raw_candidate_rows,deduplicated_station_codes,major_station_review_rows
0,13,7,7


Findings\. The candidate set is now fully under control\. We started with 13 raw rows, collapsed them to 7 station codes, and every remaining station is a major station we would plausibly consider for the final ingest\.

> 🔹 Decision: Keep the first\-pass weather ingest simple\. We will carry forward Central Park, JFK, LaGuardia, and Newark as the final station set for raw ingest, and leave Linden, Teterboro, and Caldwell as second\-pass options if we decide we need broader regional weather coverage later\.

### Build and Inspect Final Station\-Year Manifest

We’ve already picked the stations, so now we just need to expand that set across the study years and make sure the final station\-year download plan looks right\. This section builds the full manifest we’ll use for raw hourly weather ingest and gives us one last chance to catch duplicates or weird paths before we start downloading files\.

We’ll build the final station\-year manifest from the deduplicated station table and the chosen station codes\. This gives us one row per station\-year file, with the exact NOAA endpoint and local target path we expect to use\.

In [24]:
# --------------------------------------------------
# 1.1.7.3.5 Build final station-year manifest
# --------------------------------------------------

final_station_codes = [
    "USW00094728",  # NY CITY CENTRAL PARK
    "USW00094789",  # JFK INTERNATIONAL AIRPORT
    "USW00014732",  # LAGUARDIA AIRPORT
    "USW00014734",  # NEWARK LIBERTY INTERNATIONAL AIRPORT
]

final_weather_stations_df = (
    station_code_dedup_df[
        station_code_dedup_df["station_code"].isin(final_station_codes)
    ]
    .copy()
    .sort_values("name")
    .reset_index(drop=True)
)

study_years = [2023, 2024, 2025, 2026]

final_manifest_rows = []

for _, row in final_weather_stations_df.iterrows():
    for year in study_years:
        final_manifest_rows.append(
            {
                "station_code": row["station_code"],
                "station_name": row["name"],
                "year": year,
                "ghcnh_url": ghcnh_station_year_url(row["station_code"], year),
                "target_path": str(ghcnh_station_year_target_path(row["station_code"], year)),
            }
        )

final_station_year_manifest_df = (
    pd.DataFrame(final_manifest_rows)
    .drop_duplicates(subset=["station_code", "year"])
    .sort_values(["station_name", "year"])
    .reset_index(drop=True)
)

display(final_station_year_manifest_df)

,station_code,station_name,year,ghcnh_url,target_path
0,USW00094789,"JFK INTERNATIONAL AIRPORT, NY US",2023,https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2023/psv/GHCNh_USW00094789_2023.psv,source_data/weather/noaa/ghcnh/station_year/GHCNh_USW00094789_2023.psv
1,USW00094789,"JFK INTERNATIONAL AIRPORT, NY US",2024,https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2024/psv/GHCNh_USW00094789_2024.psv,source_data/weather/noaa/ghcnh/station_year/GHCNh_USW00094789_2024.psv
2,USW00094789,"JFK INTERNATIONAL AIRPORT, NY US",2025,https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2025/psv/GHCNh_USW00094789_2025.psv,source_data/weather/noaa/ghcnh/station_year/GHCNh_USW00094789_2025.psv
3,USW00094789,"JFK INTERNATIONAL AIRPORT, NY US",2026,https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2026/psv/GHCNh_USW00094789_2026.psv,source_data/weather/noaa/ghcnh/station_year/GHCNh_USW00094789_2026.psv
4,USW00014732,"LAGUARDIA AIRPORT, NY US",2023,https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2023/psv/GHCNh_USW00014732_2023.psv,source_data/weather/noaa/ghcnh/station_year/GHCNh_USW00014732_2023.psv
5,USW00014732,"LAGUARDIA AIRPORT, NY US",2024,https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2024/psv/GHCNh_USW00014732_2024.psv,source_data/weather/noaa/ghcnh/station_year/GHCNh_USW00014732_2024.psv
6,USW00014732,"LAGUARDIA AIRPORT, NY US",2025,https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2025/psv/GHCNh_USW00014732_2025.psv,source_data/weather/noaa/ghcnh/station_year/GHCNh_USW00014732_2025.psv
7,USW00014732,"LAGUARDIA AIRPORT, NY US",2026,https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2026/psv/GHCNh_USW00014732_2026.psv,source_data/weather/noaa/ghcnh/station_year/GHCNh_USW00014732_2026.psv
8,USW00014734,"NEWARK LIBERTY INTERNATIONAL AIRPORT, NJ US",2023,https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2023/psv/GHCNh_USW00014734_2023.psv,source_data/weather/noaa/ghcnh/station_year/GHCNh_USW00014734_2023.psv
9,USW00014734,"NEWARK LIBERTY INTERNATIONAL AIRPORT, NJ US",2024,https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2024/psv/GHCNh_USW00014734_2024.psv,source_data/weather/noaa/ghcnh/station_year/GHCNh_USW00014734_2024.psv


Findings\. The final station\-year manifest looks clean\. We now have one deterministic NOAA endpoint and one local target path for each station\-year file across the four chosen stations and four study years, with no leftover duplication from the earlier WBAN: / GHCND:USW\.\.\. station\-ID overlap\.

Next we’ll preview the final manifest and print a compact summary so we can confirm the station\-year plan looks clean before we move on to raw file downloads\.

In [25]:
# --------------------------------------------------
# 1.1.7.3.6 Preview and summarize final manifest
# --------------------------------------------------

display(final_station_year_manifest_df.head(20))

final_manifest_summary_df = pd.DataFrame(
    {
        "final_station_count": [final_station_year_manifest_df["station_code"].nunique()],
        "final_manifest_row_count": [len(final_station_year_manifest_df)],
        "study_years": [sorted(final_station_year_manifest_df["year"].unique().tolist())],
    }
)

display(final_manifest_summary_df)

print("Final manifest endpoints:")
for url in final_station_year_manifest_df["ghcnh_url"].tolist():
    print(url)

,station_code,station_name,year,ghcnh_url,target_path
0,USW00094789,"JFK INTERNATIONAL AIRPORT, NY US",2023,https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2023/psv/GHCNh_USW00094789_2023.psv,source_data/weather/noaa/ghcnh/station_year/GHCNh_USW00094789_2023.psv
1,USW00094789,"JFK INTERNATIONAL AIRPORT, NY US",2024,https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2024/psv/GHCNh_USW00094789_2024.psv,source_data/weather/noaa/ghcnh/station_year/GHCNh_USW00094789_2024.psv
2,USW00094789,"JFK INTERNATIONAL AIRPORT, NY US",2025,https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2025/psv/GHCNh_USW00094789_2025.psv,source_data/weather/noaa/ghcnh/station_year/GHCNh_USW00094789_2025.psv
3,USW00094789,"JFK INTERNATIONAL AIRPORT, NY US",2026,https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2026/psv/GHCNh_USW00094789_2026.psv,source_data/weather/noaa/ghcnh/station_year/GHCNh_USW00094789_2026.psv
4,USW00014732,"LAGUARDIA AIRPORT, NY US",2023,https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2023/psv/GHCNh_USW00014732_2023.psv,source_data/weather/noaa/ghcnh/station_year/GHCNh_USW00014732_2023.psv
5,USW00014732,"LAGUARDIA AIRPORT, NY US",2024,https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2024/psv/GHCNh_USW00014732_2024.psv,source_data/weather/noaa/ghcnh/station_year/GHCNh_USW00014732_2024.psv
6,USW00014732,"LAGUARDIA AIRPORT, NY US",2025,https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2025/psv/GHCNh_USW00014732_2025.psv,source_data/weather/noaa/ghcnh/station_year/GHCNh_USW00014732_2025.psv
7,USW00014732,"LAGUARDIA AIRPORT, NY US",2026,https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2026/psv/GHCNh_USW00014732_2026.psv,source_data/weather/noaa/ghcnh/station_year/GHCNh_USW00014732_2026.psv
8,USW00014734,"NEWARK LIBERTY INTERNATIONAL AIRPORT, NJ US",2023,https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2023/psv/GHCNh_USW00014734_2023.psv,source_data/weather/noaa/ghcnh/station_year/GHCNh_USW00014734_2023.psv
9,USW00014734,"NEWARK LIBERTY INTERNATIONAL AIRPORT, NJ US",2024,https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2024/psv/GHCNh_USW00014734_2024.psv,source_data/weather/noaa/ghcnh/station_year/GHCNh_USW00014734_2024.psv


,final_station_count,final_manifest_row_count,study_years
0,4,16,"[2023, 2024, 2025, 2026]"


Final manifest endpoints:
https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2023/psv/GHCNh_USW00094789_2023.psv
https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2024/psv/GHCNh_USW00094789_2024.psv
https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2025/psv/GHCNh_USW00094789_2025.psv
https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2026/psv/GHCNh_USW00094789_2026.psv
https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2023/psv/GHCNh_USW00014732_2023.psv
https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2024/psv/GHCNh_USW00014732_2024.psv
https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2025/psv/GHCNh_USW00014732_2025.psv
https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2026/psv/G

Findings\. The final download plan is now fully scoped and easy to reason about: 4 stations × 4 years = 16 raw hourly files\. The endpoint pattern is consistent across all rows, the local target paths are clean, and the manifest is small enough that the raw ingest step should stay manageable\.

### Load Raw Station\-Year Files and Validate Ingest

We’ve already locked the station set and built the final manifest, so now we just need to pull the raw station\-year files into the project and confirm that the ingest completed cleanly\. We’ll keep this mechanical: download the files, track progress and status, skip files that already exist unless we force a rebuild, and then check that the expected inventory is actually on disk and readable\.

We’ll start by defining the raw weather load toggle and the helper that downloads a station\-year file only when needed\. A file should download if we force a rebuild or if the target file does not exist yet\. This keeps the raw ingest resumable and avoids re\-downloading files that are already on disk\.

In [26]:
# --------------------------------------------------
# 1.1.7.3.7 Define raw weather load toggle and resumable download helper
# --------------------------------------------------

RUN_FULL_WEATHER_LOAD = False
FORCE_REBUILD_WEATHER_RAW = False

RAW_GHCNH_DIR = Path("source_data/weather/noaa/ghcnh/station_year")
RAW_GHCNH_DIR.mkdir(parents=True, exist_ok=True)


def should_download_file(target_path: Path, force_rebuild: bool = False) -> bool:
    return force_rebuild or (not target_path.exists())


def download_file_with_resume_logic(url: str, target_path: Path) -> dict:
    target_path.parent.mkdir(parents=True, exist_ok=True)

    try:
        resp = requests.get(url, timeout=120)
        resp.raise_for_status()
        target_path.write_bytes(resp.content)

        return {
            "download_status": "downloaded",
            "file_exists": target_path.exists(),
            "file_size_bytes": target_path.stat().st_size if target_path.exists() else None,
            "error_message": None,
        }
    except Exception as e:
        return {
            "download_status": "failed",
            "file_exists": target_path.exists(),
            "file_size_bytes": target_path.stat().st_size if target_path.exists() else None,
            "error_message": str(e),
        }

We’ll run the raw file ingest against the final manifest and decide file\-by\-file whether a download is needed\. A file should download if we force a rebuild or if the target file is missing\. That makes the load resumable and lets us rerun the cell safely after a partial failure\.

In [27]:
# --------------------------------------------------
# 1.1.7.3.8 Download raw station-year files with progress and restart safety
# --------------------------------------------------

raw_weather_ingest_rows = []

total_files = len(final_station_year_manifest_df)

for i, row in final_station_year_manifest_df.reset_index(drop=True).iterrows():
    station_code = row["station_code"]
    station_name = row["station_name"]
    year = int(row["year"])
    url = row["ghcnh_url"]
    target_path = Path(row["target_path"])

    print(f"[{i + 1}/{total_files}] {station_code} {year}")

    if should_download_file(target_path, force_rebuild=FORCE_REBUILD_WEATHER_RAW):
        result = download_file_with_resume_logic(url=url, target_path=target_path)
    else:
        result = {
            "download_status": "skipped_existing",
            "file_exists": target_path.exists(),
            "file_size_bytes": target_path.stat().st_size if target_path.exists() else None,
            "error_message": None,
        }

    raw_weather_ingest_rows.append(
        {
            "station_code": station_code,
            "station_name": station_name,
            "year": year,
            "ghcnh_url": url,
            "target_path": str(target_path),
            **result,
        }
    )

raw_weather_ingest_status_df = pd.DataFrame(raw_weather_ingest_rows)
display(raw_weather_ingest_status_df)

[1/16] USW00094789 2023
[2/16] USW00094789 2024
[3/16] USW00094789 2025
[4/16] USW00094789 2026
[5/16] USW00014732 2023
[6/16] USW00014732 2024
[7/16] USW00014732 2025
[8/16] USW00014732 2026
[9/16] USW00014734 2023
[10/16] USW00014734 2024
[11/16] USW00014734 2025
[12/16] USW00014734 2026
[13/16] USW00094728 2023
[14/16] USW00094728 2024
[15/16] USW00094728 2025
[16/16] USW00094728 2026


,station_code,station_name,year,ghcnh_url,target_path,download_status,file_exists,file_size_bytes,error_message
0,USW00094789,"JFK INTERNATIONAL AIRPORT, NY US",2023,https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2023/psv/GHCNh_USW00094789_2023.psv,source_data/weather/noaa/ghcnh/station_year/GHCNh_USW00094789_2023.psv,skipped_existing,True,12372249,None
1,USW00094789,"JFK INTERNATIONAL AIRPORT, NY US",2024,https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2024/psv/GHCNh_USW00094789_2024.psv,source_data/weather/noaa/ghcnh/station_year/GHCNh_USW00094789_2024.psv,skipped_existing,True,11796626,None
2,USW00094789,"JFK INTERNATIONAL AIRPORT, NY US",2025,https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2025/psv/GHCNh_USW00094789_2025.psv,source_data/weather/noaa/ghcnh/station_year/GHCNh_USW00094789_2025.psv,skipped_existing,True,11483981,None
3,USW00094789,"JFK INTERNATIONAL AIRPORT, NY US",2026,https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2026/psv/GHCNh_USW00094789_2026.psv,source_data/weather/noaa/ghcnh/station_year/GHCNh_USW00094789_2026.psv,skipped_existing,True,15999742,None
4,USW00014732,"LAGUARDIA AIRPORT, NY US",2023,https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2023/psv/GHCNh_USW00014732_2023.psv,source_data/weather/noaa/ghcnh/station_year/GHCNh_USW00014732_2023.psv,skipped_existing,True,12609080,None
5,USW00014732,"LAGUARDIA AIRPORT, NY US",2024,https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2024/psv/GHCNh_USW00014732_2024.psv,source_data/weather/noaa/ghcnh/station_year/GHCNh_USW00014732_2024.psv,skipped_existing,True,12466842,None
6,USW00014732,"LAGUARDIA AIRPORT, NY US",2025,https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2025/psv/GHCNh_USW00014732_2025.psv,source_data/weather/noaa/ghcnh/station_year/GHCNh_USW00014732_2025.psv,skipped_existing,True,11721392,None
7,USW00014732,"LAGUARDIA AIRPORT, NY US",2026,https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2026/psv/GHCNh_USW00014732_2026.psv,source_data/weather/noaa/ghcnh/station_year/GHCNh_USW00014732_2026.psv,skipped_existing,True,18329832,None
8,USW00014734,"NEWARK LIBERTY INTERNATIONAL AIRPORT, NJ US",2023,https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2023/psv/GHCNh_USW00014734_2023.psv,source_data/weather/noaa/ghcnh/station_year/GHCNh_USW00014734_2023.psv,skipped_existing,True,11595526,None
9,USW00014734,"NEWARK LIBERTY INTERNATIONAL AIRPORT, NJ US",2024,https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2024/psv/GHCNh_USW00014734_2024.psv,source_data/weather/noaa/ghcnh/station_year/GHCNh_USW00014734_2024.psv,skipped_existing,True,12356934,None


Findings\. The raw weather ingest worked cleanly\. All 16 expected station\-year files are now on disk, 15 were newly downloaded in this run, and the one Central Park 2023 file we had already pulled was correctly skipped rather than re\-downloaded\. File sizes look healthy across the board and there were no download failures\.

We’ll now validate the raw file inventory so we can see whether the expected station\-year files are actually on disk and whether any downloads failed or produced suspiciously small files\.

In [28]:
# --------------------------------------------------
# 1.1.7.3.9 Validate raw file inventory
# --------------------------------------------------

if raw_weather_ingest_status_df.empty:
    print("No raw weather ingest results to validate yet.")
else:
    raw_file_inventory_summary_df = pd.DataFrame(
        {
            "expected_file_count": [len(final_station_year_manifest_df)],
            "status_row_count": [len(raw_weather_ingest_status_df)],
            "downloaded_count": [(raw_weather_ingest_status_df["download_status"] == "downloaded").sum()],
            "skipped_existing_count": [(raw_weather_ingest_status_df["download_status"] == "skipped_existing").sum()],
            "failed_count": [(raw_weather_ingest_status_df["download_status"] == "failed").sum()],
            "file_exists_count": [raw_weather_ingest_status_df["file_exists"].sum()],
            "zero_byte_file_count": [(raw_weather_ingest_status_df["file_size_bytes"].fillna(0) == 0).sum()],
        }
    )

    display(raw_file_inventory_summary_df)

    failed_or_missing_df = raw_weather_ingest_status_df[
        (raw_weather_ingest_status_df["download_status"] == "failed")
        | (~raw_weather_ingest_status_df["file_exists"])
        | (raw_weather_ingest_status_df["file_size_bytes"].fillna(0) == 0)
    ].copy()

    display(failed_or_missing_df)

,expected_file_count,status_row_count,downloaded_count,skipped_existing_count,failed_count,file_exists_count,zero_byte_file_count
0,16,16,0,16,0,16,0


,station_code,station_name,year,ghcnh_url,target_path,download_status,file_exists,file_size_bytes,error_message


Findings\. The inventory check came back clean\. The expected file count matches the status table exactly, all 16 files exist, there were no failures, and none of the files are zero\-byte placeholders\. So the raw hourly weather source is now fully loaded and ready for the standardization step\.

\-\-\-

## 1\.1\.7\.4 Coverage, Missingness, and Measurement Quality

This section stays at the raw NOAA layer and checks whether the loaded hourly weather data is complete and coherent enough to hand off\. We are not standardizing, bucketing, or assigning geography here\. Instead, we surface station coverage, timestamp continuity, missingness, duplicates, and basic measurement sanity so any source\-quality issues are visible before downstream staging and modeling\.

\*\*1\.1\.7\.4\.1 Coverage and timestamp continuity\.\*\*
This block reads the raw station\-year files directly, aligns station names from the station manifest when needed, and checks whether the hourly timestamps cover the study window in a way that is usable for staging\.

In [29]:
# ---------------------------------------------------------------------
# 1.1.7.4.1 Coverage and timestamp continuity
# ---------------------------------------------------------------------

def qa_first_present_column(columns, candidates):
    column_set = set(columns)
    for candidate in candidates:
        if candidate in column_set:
            return candidate
    return None


def qa_collect_raw_weather_files(root: Path) -> List[Path]:
    if not root.exists():
        return []
    files = [path for path in root.rglob("*.psv") if path.is_file()]
    return sorted(files)


def qa_load_psv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, sep="|", low_memory=False)


raw_weather_files = qa_collect_raw_weather_files(WEATHER_RAW_DOWNLOAD_DIR)
if not raw_weather_files:
    raise FileNotFoundError(
        f"No NOAA weather files were found in {WEATHER_RAW_DOWNLOAD_DIR} for QA."
    )

print_step(f"Loading {len(raw_weather_files)} raw NOAA station-year files for QA.")
print_step(f"Raw file preview: {[path.name for path in raw_weather_files[:6]]}")

raw_weather_frames = [qa_load_psv(path) for path in raw_weather_files]
weather_qa_source_df = pd.concat(raw_weather_frames, ignore_index=True, copy=False)

if weather_qa_source_df.empty:
    raise AssertionError("Weather QA source dataframe is empty.")

print_step(f"QA source columns: {list(weather_qa_source_df.columns)}")

weather_qa_station_col = qa_first_present_column(
    weather_qa_source_df.columns,
    ["weather_station_id", "station_id", "station", "station_code", "STATION", "STATION_ID"],
)
weather_qa_datetime_col = qa_first_present_column(
    weather_qa_source_df.columns,
    ["observation_datetime", "datetime", "timestamp", "DATE", "date", "date_time", "DATE_TIME"],
)
weather_qa_station_name_col = qa_first_present_column(
    weather_qa_source_df.columns,
    ["weather_station_name", "station_name", "source_station_name", "NAME", "name"],
)

if weather_qa_station_col is None or weather_qa_datetime_col is None:
    raise AssertionError(
        "Weather QA requires station and timestamp columns, but one or both were missing."
    )

weather_station_lookup_df = pd.concat(
    [
        pilot_weather_station_config_df[["weather_station_id", "weather_station_name", "source_station_id", "source_station_name"]],
        full_weather_station_config_df[["weather_station_id", "weather_station_name", "source_station_id", "source_station_name"]],
    ],
    ignore_index=True,
).drop_duplicates(subset=["weather_station_id"], keep="first")

weather_station_name_lookup = dict(
    zip(
        weather_station_lookup_df["weather_station_id"].astype(str),
        weather_station_lookup_df["weather_station_name"].astype(str),
    )
)

weather_qa_frame = weather_qa_source_df.copy()
weather_qa_frame["weather_station_id"] = weather_qa_frame[weather_qa_station_col].astype("string")
weather_qa_frame["observation_datetime"] = pd.to_datetime(
    weather_qa_frame[weather_qa_datetime_col], errors="coerce"
)
weather_qa_frame["observation_hour"] = weather_qa_frame["observation_datetime"].dt.floor("h")

if weather_qa_station_name_col is not None:
    weather_qa_frame["weather_station_name"] = weather_qa_frame[weather_qa_station_name_col].astype("string")
else:
    weather_qa_frame["weather_station_name"] = pd.NA

weather_qa_frame["weather_station_name"] = weather_qa_frame["weather_station_name"].fillna(
    weather_qa_frame["weather_station_id"].astype(str).map(weather_station_name_lookup)
)
weather_qa_frame["weather_station_name"] = weather_qa_frame["weather_station_name"].astype("string")

weather_qa_frame["in_study_period"] = (
    (weather_qa_frame["observation_datetime"] >= STUDY_START_TIMESTAMP)
    & (weather_qa_frame["observation_datetime"] < STUDY_END_TIMESTAMP_EXCLUSIVE)
)
weather_qa_frame["in_study_hour"] = (
    (weather_qa_frame["observation_hour"] >= STUDY_START_TIMESTAMP)
    & (weather_qa_frame["observation_hour"] < STUDY_END_TIMESTAMP_EXCLUSIVE)
)

study_expected_hours = int(
    (STUDY_END_TIMESTAMP_EXCLUSIVE - STUDY_START_TIMESTAMP) / pd.Timedelta(hours=1)
)


def qa_hours_between(min_ts, max_ts):
    if pd.isna(min_ts) or pd.isna(max_ts):
        return pd.NA
    return int((max_ts - min_ts) / pd.Timedelta(hours=1)) + 1


weather_qa_station_coverage_df = (
    weather_qa_frame.groupby(["weather_station_id", "weather_station_name"], dropna=False)
    .agg(
        raw_row_count=("observation_datetime", "size"),
        missing_observation_datetime_rows=("observation_datetime", lambda s: int(s.isna().sum())),
        raw_distinct_observation_hours=("observation_hour", "nunique"),
        study_distinct_observation_hours=("observation_hour", lambda s: int(s[weather_qa_frame.loc[s.index, "in_study_hour"]].nunique())),
        min_observation_datetime=("observation_datetime", "min"),
        max_observation_datetime=("observation_datetime", "max"),
        out_of_study_period_rows=("in_study_period", lambda s: int((~s).sum())),
    )
    .reset_index()
)

weather_qa_station_coverage_df["distinct_observation_hours"] = weather_qa_station_coverage_df[
    "study_distinct_observation_hours"
]
weather_qa_station_coverage_df["expected_study_hours"] = study_expected_hours
weather_qa_station_coverage_df["observed_span_hours"] = weather_qa_station_coverage_df.apply(
    lambda row: qa_hours_between(row["min_observation_datetime"], row["max_observation_datetime"]),
    axis=1,
)
weather_qa_station_coverage_df["coverage_pct_vs_study"] = (
    weather_qa_station_coverage_df["distinct_observation_hours"]
    / weather_qa_station_coverage_df["expected_study_hours"]
    * 100
).round(4)
weather_qa_station_coverage_df["coverage_pct_vs_span"] = (
    weather_qa_station_coverage_df["raw_distinct_observation_hours"]
    / weather_qa_station_coverage_df["observed_span_hours"].replace({0: pd.NA})
    * 100
).round(4)

weather_qa_station_coverage_df = weather_qa_station_coverage_df.sort_values(
    ["raw_row_count", "weather_station_id"], ascending=[False, True]
).reset_index(drop=True)

weather_qa_coverage_overall_df = pd.DataFrame(
    {
        "metric": [
            "station_count",
            "raw_row_count",
            "raw_distinct_observation_hours",
            "study_distinct_observation_hours",
            "missing_observation_datetime_rows",
            "out_of_study_period_rows",
            "study_expected_hours",
        ],
        "value": [
            weather_qa_station_coverage_df["weather_station_id"].nunique(dropna=True),
            int(weather_qa_station_coverage_df["raw_row_count"].sum()),
            int(weather_qa_station_coverage_df["raw_distinct_observation_hours"].sum()),
            int(weather_qa_station_coverage_df["study_distinct_observation_hours"].sum()),
            int(weather_qa_station_coverage_df["missing_observation_datetime_rows"].sum()),
            int(weather_qa_station_coverage_df["out_of_study_period_rows"].sum()),
            study_expected_hours,
        ],
    }
)

# Keep downstream blocks aligned with the historical names.
weather_qa_station_coverage_df["row_count"] = weather_qa_station_coverage_df["raw_row_count"]

print_step("Weather QA coverage and timestamp continuity ready.")
display(weather_qa_coverage_overall_df)
display(weather_qa_station_coverage_df)

[1.1.7] Loading 16 raw NOAA station-year files for QA.
[1.1.7] Raw file preview: ['GHCNh_USW00014732_2023.psv', 'GHCNh_USW00014732_2024.psv', 'GHCNh_USW00014732_2025.psv', 'GHCNh_USW00014732_2026.psv', 'GHCNh_USW00014734_2023.psv', 'GHCNh_USW00014734_2024.psv']
[1.1.7] QA source columns: ['STATION', 'Station_name', 'DATE', 'Year', 'Month', 'Day', 'Hour', 'Minute', 'LATITUDE', 'LONGITUDE', 'ELEVATION', 'temperature', 'temperature_Measurement_Code', 'temperature_Quality_Code', 'temperature_Report_Type', 'temperature_Source_Code', 'temperature_Source_Station_ID', 'dew_point_temperature', 'dew_point_temperature_Measurement_Code', 'dew_point_temperature_Quality_Code', 'dew_point_temperature_Report_Type', 'dew_point_temperature_Source_Code', 'dew_point_temperature_Source_Station_ID', 'station_level_pressure', 'station_level_pressure_Measurement_Code', 'station_level_pressure_Quality_Code', 'station_level_pressure_Report_Type', 'station_level_pressure_Source_Code', 'station_level_pressure_Sou

,metric,value
0,station_count,4
1,raw_row_count,318349
2,raw_distinct_observation_hours,122601
3,study_distinct_observation_hours,113325
4,missing_observation_datetime_rows,0
5,out_of_study_period_rows,96410
6,study_expected_hours,28464


,weather_station_id,weather_station_name,raw_row_count,missing_observation_datetime_rows,raw_distinct_observation_hours,study_distinct_observation_hours,min_observation_datetime,max_observation_datetime,out_of_study_period_rows,distinct_observation_hours,expected_study_hours,observed_span_hours,coverage_pct_vs_study,coverage_pct_vs_span,row_count
0,USW00014734,Newark Liberty International Airport,88175,0,30668,28348,2023-01-01 00:51:00,2026-07-06 15:51:00,27423,28348,28464,30784,99.5925,99.6232,88175
1,USW00094728,New York Central Park,81328,0,30613,28293,2023-01-01 00:06:00,2026-07-06 15:51:00,24711,28293,28464,30784,99.3992,99.4445,81328
2,USW00014732,New York LaGuardia Airport,77457,0,30662,28344,2023-01-01 00:15:00,2026-07-06 15:51:00,20640,28344,28464,30784,99.5784,99.6037,77457
3,USW00094789,New York JFK Airport,71389,0,30658,28340,2023-01-01 00:00:00,2026-07-06 15:51:00,23636,28340,28464,30784,99.5644,99.5907,71389


Findings\. The raw weather pull is complete across four stations with no missing timestamps and no duplicate station\-hour rows\. The only issue is scope leakage: 96,410 rows fall outside the study window, because the source files extend past the March 31, 2026 cutoff\. So the feed looks structurally solid, but it still needs the study\-period filter enforced before staging\.

A quick schema inventory first: the QA below is now going to use the actual columns in \`weather\_qa\_frame\`, not a guessed target schema\. That lets us separate true missingness from optional reporting, categorical weather descriptors, and sentinel codes before we summarize the table\.

In [30]:
# Diagnostic block: list out schema

actual_columns = list(weather_qa_frame.columns)

core_numeric_weather_fields = [
    "temperature",
    "dew_point_temperature",
    "station_level_pressure",
    "sea_level_pressure",
    "wind_direction",
    "wind_speed",
    "wind_gust",
    "precipitation",
    "relative_humidity",
    "wet_bulb_temperature",
    "snow_depth",
    "visibility",
    "altimeter",
    "pressure_3hr_change",
    "ceiling_height",
]

categorical_present_weather_fields = [
    "sky_condition",
    "pres_wx_MW1", "pres_wx_MW2", "pres_wx_MW3",
    "pres_wx_AU1", "pres_wx_AU2", "pres_wx_AU3",
    "pres_wx_AW1", "pres_wx_AW2", "pres_wx_AW3",
]

code_or_metadata_fields = [
    c for c in actual_columns
    if c.endswith(("_Measurement_Code", "_Quality_Code", "_Report_Type", "_Source_Code", "_Source_Station_ID"))
]

geo_time_id_fields = [
    "STATION", "Station_name", "DATE", "Year", "Month", "Day", "Hour", "Minute",
    "LATITUDE", "LONGITUDE", "ELEVATION",
    "weather_station_id", "weather_station_name", "observation_datetime",
    "observation_hour", "in_study_period", "in_study_hour",
]

group_lookup = {}
for c in actual_columns:
    if c in core_numeric_weather_fields:
        group_lookup[c] = "core_numeric_weather"
    elif c in categorical_present_weather_fields:
        group_lookup[c] = "categorical_present_weather"
    elif c in code_or_metadata_fields:
        group_lookup[c] = "code_or_metadata"
    elif c in geo_time_id_fields:
        group_lookup[c] = "geo_time_or_ids"
    else:
        group_lookup[c] = "other"

weather_qa_schema_inventory_df = pd.DataFrame(
    {"column": actual_columns, "group": [group_lookup[c] for c in actual_columns]}
)

print("Core numeric weather fields present:")
print([c for c in core_numeric_weather_fields if c in actual_columns])

print("\nCategorical / present-weather fields present:")
print([c for c in categorical_present_weather_fields if c in actual_columns])

print("\nCode / metadata fields present:")
print(code_or_metadata_fields)

display(weather_qa_schema_inventory_df)

Core numeric weather fields present:
['temperature', 'dew_point_temperature', 'station_level_pressure', 'sea_level_pressure', 'wind_direction', 'wind_speed', 'wind_gust', 'precipitation', 'relative_humidity', 'wet_bulb_temperature', 'snow_depth', 'visibility', 'altimeter', 'pressure_3hr_change', 'ceiling_height']

Categorical / present-weather fields present:
['sky_condition', 'pres_wx_MW1', 'pres_wx_MW2', 'pres_wx_MW3', 'pres_wx_AU1', 'pres_wx_AU2', 'pres_wx_AU3', 'pres_wx_AW1', 'pres_wx_AW2', 'pres_wx_AW3']

Code / metadata fields present:
['temperature_Measurement_Code', 'temperature_Quality_Code', 'temperature_Report_Type', 'temperature_Source_Code', 'temperature_Source_Station_ID', 'dew_point_temperature_Measurement_Code', 'dew_point_temperature_Quality_Code', 'dew_point_temperature_Report_Type', 'dew_point_temperature_Source_Code', 'dew_point_temperature_Source_Station_ID', 'station_level_pressure_Measurement_Code', 'station_level_pressure_Quality_Code', 'station_level_pressure_R

,column,group
0,STATION,geo_time_or_ids
1,Station_name,geo_time_or_ids
2,DATE,geo_time_or_ids
3,Year,geo_time_or_ids
4,Month,geo_time_or_ids
...,...,...
330,observation_datetime,geo_time_or_ids
331,observation_hour,geo_time_or_ids
332,weather_station_name,geo_time_or_ids
333,in_study_period,geo_time_or_ids


Raw NOAA / GHCNh weather tables are sparse by design\. Many fields are optional, report\-type dependent, or only populated when a station emits that measurement\. High null rates in the standardized weather frame do not automatically mean the source is broken; they often reflect source sparsity, optional present\-weather fields, or sentinel artifacts\. This QA is meant to separate real missingness from schema mismatch and sparse reporting\.

In [31]:
# ---------------------------------------------------------------------
# 1.1.7.4.2 Missingness and measurement quality
# ---------------------------------------------------------------------

def _top_values(series, n=3):
    values = series.dropna().astype("string")
    if values.empty:
        return pd.NA
    counts = values.value_counts().head(n)
    return "; ".join(f"{idx} ({count})" for idx, count in counts.items())

def _sentinel_mask(raw_series, sentinel_values):
    mask = pd.Series(False, index=raw_series.index)
    if not sentinel_values:
        return mask
    raw_text = raw_series.astype("string")
    for sentinel in sentinel_values:
        mask |= raw_series.eq(sentinel) | raw_text.eq(str(sentinel))
    return mask.fillna(False)

def _numeric_summary(frame, measurement_label, source_column, lower_bound, upper_bound, sentinel_values=None):
    raw = frame[source_column]
    numeric = pd.to_numeric(raw, errors="coerce")
    non_null_mask = raw.notna()
    coerced_numeric_null_count = int(non_null_mask.sum() - numeric.notna().sum())
    sentinel_value_count = int(_sentinel_mask(raw, sentinel_values or []).sum())
    out_of_bounds_mask = numeric.notna() & ((numeric < lower_bound) | (numeric > upper_bound))

    return {
        "measurement_label": measurement_label,
        "source_column": source_column,
        "field_type": "numeric",
        "present": True,
        "row_count": int(len(frame)),
        "null_count": int(raw.isna().sum()),
        "null_pct": round(100 * raw.isna().mean(), 4),
        "distinct_non_null_value_count": int(numeric.dropna().nunique()),
        "coerced_numeric_null_count": int(coerced_numeric_null_count),
        "min_value": float(numeric.min()) if numeric.notna().any() else pd.NA,
        "median_value": float(numeric.median()) if numeric.notna().any() else pd.NA,
        "max_value": float(numeric.max()) if numeric.notna().any() else pd.NA,
        "sentinel_value_count": int(sentinel_value_count),
        "out_of_bounds_count": int((out_of_bounds_mask | _sentinel_mask(raw, sentinel_values or [])).sum()),
        "top_non_null_values": pd.NA,
    }

def _categorical_summary(frame, measurement_label, source_column):
    raw = frame[source_column]
    non_null = raw.dropna()
    return {
        "measurement_label": measurement_label,
        "source_column": source_column,
        "field_type": "categorical",
        "present": True,
        "row_count": int(len(frame)),
        "null_count": int(raw.isna().sum()),
        "null_pct": round(100 * raw.isna().mean(), 4),
        "distinct_non_null_value_count": int(non_null.astype("string").nunique()),
        "coerced_numeric_null_count": pd.NA,
        "min_value": pd.NA,
        "median_value": pd.NA,
        "max_value": pd.NA,
        "sentinel_value_count": pd.NA,
        "out_of_bounds_count": pd.NA,
        "top_non_null_values": _top_values(raw),
    }

measurement_specs = [
    ("dry_bulb_temperature", "temperature", "numeric", -80, 60, []),
    ("dew_point_temperature", "dew_point_temperature", "numeric", -90, 50, []),
    ("station_level_pressure", "station_level_pressure", "numeric", 850, 1100, []),
    ("sea_level_pressure", "sea_level_pressure", "numeric", 850, 1100, []),
    ("wind_direction", "wind_direction", "numeric", 0, 360, [999]),
    ("wind_speed", "wind_speed", "numeric", 0, 200, []),
    ("wind_gust", "wind_gust", "numeric", 0, 250, []),
    ("precipitation", "precipitation", "numeric", 0, 500, []),
    ("relative_humidity", "relative_humidity", "numeric", 0, 100, []),
    ("wet_bulb_temperature", "wet_bulb_temperature", "numeric", -80, 60, []),
    ("snow_depth", "snow_depth", "numeric", 0, 1000, []),
    ("visibility", "visibility", "numeric", 0, 200, []),
    ("altimeter", "altimeter", "numeric", 850, 1100, []),
    ("pressure_3hr_change", "pressure_3hr_change", "numeric", -100, 100, []),
    ("ceiling_height", "ceiling_height", "numeric", 0, 50000, []),
    ("sky_condition", "sky_condition", "categorical", None, None, None),
    ("pres_wx_MW1", "pres_wx_MW1", "categorical", None, None, None),
    ("pres_wx_MW2", "pres_wx_MW2", "categorical", None, None, None),
    ("pres_wx_MW3", "pres_wx_MW3", "categorical", None, None, None),
    ("pres_wx_AU1", "pres_wx_AU1", "categorical", None, None, None),
    ("pres_wx_AU2", "pres_wx_AU2", "categorical", None, None, None),
    ("pres_wx_AU3", "pres_wx_AU3", "categorical", None, None, None),
    ("pres_wx_AW1", "pres_wx_AW1", "categorical", None, None, None),
    ("pres_wx_AW2", "pres_wx_AW2", "categorical", None, None, None),
    ("pres_wx_AW3", "pres_wx_AW3", "categorical", None, None, None),
]

qa_rows = []
for measurement_label, source_column, field_type, lower_bound, upper_bound, sentinel_values in measurement_specs:
    if source_column in weather_qa_frame.columns:
        if field_type == "numeric":
            qa_rows.append(_numeric_summary(weather_qa_frame, measurement_label, source_column, lower_bound, upper_bound, sentinel_values))
        else:
            qa_rows.append(_categorical_summary(weather_qa_frame, measurement_label, source_column))
    else:
        qa_rows.append({
            "measurement_label": measurement_label,
            "source_column": source_column,
            "field_type": field_type,
            "present": False,
            "row_count": int(len(weather_qa_frame)),
            "null_count": int(len(weather_qa_frame)),
            "null_pct": 100.0,
            "distinct_non_null_value_count": pd.NA,
            "coerced_numeric_null_count": pd.NA,
            "min_value": pd.NA,
            "median_value": pd.NA,
            "max_value": pd.NA,
            "sentinel_value_count": pd.NA,
            "out_of_bounds_count": pd.NA,
            "top_non_null_values": pd.NA,
        })

weather_qa_measurement_missingness_df = pd.DataFrame(qa_rows)

display(weather_qa_measurement_missingness_df)

,measurement_label,source_column,field_type,present,row_count,null_count,null_pct,distinct_non_null_value_count,coerced_numeric_null_count,min_value,median_value,max_value,sentinel_value_count,out_of_bounds_count,top_non_null_values
0,dry_bulb_temperature,temperature,numeric,True,318349,145112,45.5827,142,0,-22.0,13.3,40.0,0,0,<NA>
1,dew_point_temperature,dew_point_temperature,numeric,True,318349,145155,45.5962,136,0,-27.2,6.7,25.6,0,0,<NA>
2,station_level_pressure,station_level_pressure,numeric,True,318349,179536,56.3960,607,0,977.1,1013.9,1048.2,0,0,<NA>
3,sea_level_pressure,sea_level_pressure,numeric,True,318349,167399,52.5835,646,0,982.0,1016.2,1048.9,0,0,<NA>
4,wind_direction,wind_direction,numeric,True,318349,150782,47.3637,39,0,0.0,240.0,999.0,26021,26021,<NA>
5,wind_speed,wind_speed,numeric,True,318349,149615,46.9972,46,0,0.0,3.6,20.6,0,0,<NA>
6,wind_gust,wind_gust,numeric,True,318349,289723,91.0080,51,0,7.2,11.3,32.4,0,0,<NA>
7,precipitation,precipitation,numeric,True,318349,212636,66.7934,156,0,0.0,0.0,52.8,0,0,<NA>
8,relative_humidity,relative_humidity,numeric,True,318349,145155,45.5962,89,0,10.0,64.0,1156.0,0,1,<NA>
9,wet_bulb_temperature,wet_bulb_temperature,numeric,True,318349,188601,59.2435,432,0,-17.2,10.6,27.3,0,0,<NA>


Findings: The QA results show that the most useful weather signals to keep are dry\_bulb\_temperature, precipitation, visibility, wind\_speed, wind\_gust, relative\_humidity, ceiling\_height, and pressure\_3hr\_change\. altimeter, sea\_level\_pressure, station\_level\_pressure, and snow\_depth are present but feel more like secondary context than core app features\. wind\_direction needs cleanup because of the 999 sentinel, and relative\_humidity needs correction because of the impossible high value\. The sky\_condition and pres\_wx\_\* fields are real source fields, but they are very sparse and should stay as raw categorical evidence rather than headline app metrics\.

> ✅ Decision: Keep the clean, operational weather fields as primary features and disregard the sparse raw code fields as core app metrics\.

This block looks for repeated station\-hour rows and large timestamp gaps, because those are the source issues most likely to distort later staging if we do not catch them early\.

In [32]:
# ---------------------------------------------------------------------
# 1.1.7.4.3 Duplicate station-hour and gap audit
# ---------------------------------------------------------------------

duplicate_key_cols = ["weather_station_id", "observation_datetime"]
weather_qa_duplicate_station_hour_df = weather_qa_frame.loc[
    weather_qa_frame.duplicated(duplicate_key_cols, keep=False)
].copy()

if not weather_qa_duplicate_station_hour_df.empty:
    weather_qa_duplicate_station_hour_df = weather_qa_duplicate_station_hour_df.sort_values(
        duplicate_key_cols, kind="mergesort"
    ).reset_index(drop=True)
    display(weather_qa_duplicate_station_hour_df)


gap_rows = []
for station_id, station_df in weather_qa_frame.sort_values(
    ["weather_station_id", "observation_datetime"], kind="mergesort"
).groupby("weather_station_id", dropna=False):
    ordered_datetimes = (
        station_df["observation_datetime"].dropna().drop_duplicates().sort_values().reset_index(drop=True)
    )
    gap_deltas = ordered_datetimes.diff().dropna()
    gap_hours = gap_deltas.dt.total_seconds().div(3600)

    station_name = station_df["weather_station_name"].dropna().iloc[0] if station_df["weather_station_name"].notna().any() else pd.NA
    duplicate_station_hour_rows = int(station_df.duplicated(duplicate_key_cols, keep=False).sum())

    gap_rows.append(
        {
            "weather_station_id": station_id,
            "weather_station_name": station_name,
            "row_count": len(station_df),
            "duplicate_station_hour_rows": duplicate_station_hour_rows,
            "distinct_observation_hours": int(ordered_datetimes.nunique()),
            "gap_count_over_1_hour": int((gap_hours > 1).sum()) if not gap_hours.empty else 0,
            "max_gap_hours": float(gap_hours.max()) if not gap_hours.empty else pd.NA,
            "min_gap_hours": float(gap_hours.min()) if not gap_hours.empty else pd.NA,
        }
    )

weather_qa_gap_summary_df = pd.DataFrame(gap_rows).sort_values(
    ["duplicate_station_hour_rows", "gap_count_over_1_hour", "weather_station_id"],
    ascending=[False, False, True],
).reset_index(drop=True)

weather_qa_duplicate_summary_df = pd.DataFrame(
    {
        "metric": [
            "duplicate_station_hour_rows",
            "duplicate_station_hour_groups",
            "stations_with_duplicates",
        ],
        "value": [
            len(weather_qa_duplicate_station_hour_df),
            weather_qa_duplicate_station_hour_df.groupby(duplicate_key_cols, dropna=False).ngroups
            if not weather_qa_duplicate_station_hour_df.empty
            else 0,
            int((weather_qa_gap_summary_df["duplicate_station_hour_rows"] > 0).sum()),
        ],
    }
)

display(weather_qa_duplicate_summary_df)
display(weather_qa_gap_summary_df)

,metric,value
0,duplicate_station_hour_rows,0
1,duplicate_station_hour_groups,0
2,stations_with_duplicates,0


,weather_station_id,weather_station_name,row_count,duplicate_station_hour_rows,distinct_observation_hours,gap_count_over_1_hour,max_gap_hours,min_gap_hours
0,USW00094728,New York Central Park,81328,0,81328,110,91.0,0.016667
1,USW00014734,Newark Liberty International Airport,88175,0,88175,31,91.0,0.016667
2,USW00094789,New York JFK Airport,71389,0,71389,31,91.0,0.016667
3,USW00014732,New York LaGuardia Airport,77457,0,77457,29,91.0,0.016667


Findings: The duplicate and continuity QA looks healthy overall: there are no duplicate station\-hour rows, no duplicate station\-hour groups, and no stations with duplication issues\. The remaining gaps are therefore true coverage interruptions rather than accidental repeated observations, and the largest observed gap is 91 hours across all four stations\. With the station configuration now explicitly carrying Central Park alongside the three airport stations, the gap summary should also retain station names cleanly rather than leaving one station unlabeled\.

> Section Summary\. This subsection now works as the raw\-weather QA checkpoint for the notebook\. It confirms that the NOAA/GHCNh station files loaded successfully, that hourly timestamps are parseable and nearly complete across the study window, that duplicate station\-hour observations are not a problem, and that the major weather fields are real source columns rather than guessed schema placeholders\. It also helps us separate true missingness from normal source sparsity, optional present\-weather reporting, and sentinel\-value artifacts, which is the right stopping point before any later standardization or feature engineering\.

\-\-\-

## 1\.1\.7\.5 Final Staged Weather Export and Manifest

This closing section packages the weather notebook into app\-ready sidecar outputs\. The goal here is not to invent a new weather transformation layer, but to validate the final study\-period weather frame we already loaded, export the granular station observations, write a compact station manifest and QA summary, and leave behind a lightweight contract that downstream notebooks can trust\.

We first validate the final staged weather frame using the schema we actually loaded in this notebook\. This check confirms the required identifiers and timestamps are present, restricts the export to the study period, resolves station names from the best available sources, and assembles the final in\-memory tables that will be written out below\.

In [33]:
# ---------------------------------------------------------------------
# 1.1.7.5.1 Validate final staged weather frame and assemble exports
# ---------------------------------------------------------------------

weather_export_frame = weather_qa_frame.copy()
weather_export_frame["observation_datetime"] = pd.to_datetime(
    weather_export_frame["observation_datetime"],
    errors="coerce",
)
weather_export_frame["observation_date"] = weather_export_frame[
    "observation_datetime"
].dt.date
weather_export_frame["observation_hour"] = weather_export_frame[
    "observation_datetime"
].dt.floor("h")

station_name_lookup_df = (
    full_weather_station_config_df[
        ["weather_station_id", "weather_station_name"]
    ]
    .drop_duplicates(subset=["weather_station_id"])
    .reset_index(drop=True)
)

# Prefer station names carried in the QA frame, but backfill from the explicit
# notebook station config so the final export is not relying on raw-source naming.
if "weather_station_name" not in weather_export_frame.columns:
    weather_export_frame = weather_export_frame.merge(
        station_name_lookup_df,
        on="weather_station_id",
        how="left",
    )
else:
    weather_export_frame = weather_export_frame.merge(
        station_name_lookup_df.rename(
            columns={"weather_station_name": "configured_weather_station_name"}
        ),
        on="weather_station_id",
        how="left",
    )
    weather_export_frame["weather_station_name"] = weather_export_frame[
        "weather_station_name"
    ].fillna(weather_export_frame["configured_weather_station_name"])
    weather_export_frame = weather_export_frame.drop(
        columns=["configured_weather_station_name"]
    )

study_period_mask = (
    (weather_export_frame["observation_datetime"] >= STUDY_START_TIMESTAMP)
    & (weather_export_frame["observation_datetime"] < STUDY_END_TIMESTAMP_EXCLUSIVE)
)
weather_study_period_df = weather_export_frame.loc[study_period_mask].copy()

curated_weather_columns = [
    "weather_station_id",
    "weather_station_name",
    "observation_datetime",
    "observation_date",
    "observation_hour",
    "temperature",
    "precipitation",
    "visibility",
    "wind_speed",
    "wind_gust",
    "relative_humidity",
    "ceiling_height",
    "pressure_3hr_change",
]
weather_hourly_export_df = weather_study_period_df[curated_weather_columns].copy()
weather_hourly_audit_export_df = weather_study_period_df.copy()

weather_station_manifest_df = (
    weather_hourly_export_df.groupby(
        ["weather_station_id", "weather_station_name"], dropna=False
    )
    .agg(
        row_count=("observation_datetime", "size"),
        distinct_observation_hours=("observation_hour", "nunique"),
        min_observation_datetime=("observation_datetime", "min"),
        max_observation_datetime=("observation_datetime", "max"),
    )
    .reset_index()
    .sort_values("weather_station_id")
    .reset_index(drop=True)
)

coverage_columns = [
    column
    for column in [
        "weather_station_id",
        "coverage_pct_vs_study",
        "gap_count_over_1_hour",
        "max_gap_hours",
        "qa_status",
    ]
    if column in weather_qa_station_coverage_df.columns
]
weather_station_manifest_df = weather_station_manifest_df.merge(
    weather_qa_station_coverage_df[coverage_columns].drop_duplicates(
        subset=["weather_station_id"]
    ),
    on="weather_station_id",
    how="left",
)

final_weather_schema_validation_df = pd.DataFrame(
    {
        "check": [
            "required_columns_present",
            "non_null_weather_station_id",
            "non_null_observation_datetime",
            "study_period_filtered",
            "export_row_count_positive",
            "export_station_count_positive",
        ],
        "status": [
            {"weather_station_id", "weather_station_name", "observation_datetime"}.issubset(
                weather_hourly_export_df.columns
            ),
            bool(weather_hourly_export_df["weather_station_id"].notna().all()),
            bool(weather_hourly_export_df["observation_datetime"].notna().all()),
            bool(
                weather_hourly_export_df["observation_datetime"].min()
                >= STUDY_START_TIMESTAMP
            )
            and bool(
                weather_hourly_export_df["observation_datetime"].max()
                < STUDY_END_TIMESTAMP_EXCLUSIVE
            ),
            bool(len(weather_hourly_export_df) > 0),
            bool(weather_hourly_export_df["weather_station_id"].nunique() > 0),
        ],
    }
)
assert final_weather_schema_validation_df["status"].all(), (
    "Final weather schema validation failed."
)

final_weather_export_overview_df = pd.DataFrame(
    {
        "metric": [
            "study_period_row_count",
            "study_period_station_count",
            "study_period_min_observation_datetime",
            "study_period_max_observation_datetime",
        ],
        "value": [
            len(weather_hourly_export_df),
            weather_hourly_export_df["weather_station_id"].nunique(),
            weather_hourly_export_df["observation_datetime"].min(),
            weather_hourly_export_df["observation_datetime"].max(),
        ],
    }
)

print_step(
    f"Assembled curated weather export with {len(weather_hourly_export_df):,} "
    f"study-period rows across {weather_hourly_export_df['weather_station_id'].nunique()} stations."
)
display(final_weather_schema_validation_df)
display(final_weather_export_overview_df)

[1.1.7] Assembled curated weather export with 221,939 study-period rows across 4 stations.


,check,status
0,required_columns_present,True
1,non_null_weather_station_id,True
2,non_null_observation_datetime,True
3,study_period_filtered,True
4,export_row_count_positive,True
5,export_station_count_positive,True


,metric,value
0,study_period_row_count,221939
1,study_period_station_count,4
2,study_period_min_observation_datetime,2023-01-01 00:00:00
3,study_period_max_observation_datetime,2026-03-31 23:55:00


Findings: The final staged weather export passed all core validation checks cleanly\. The required fields are present, station IDs and observation timestamps are fully populated, the export is correctly restricted to the study period, and the final output has a positive row count and station count\. In total, the staged export retains 221,939 hourly weather observations across four stations, spanning the full intended study window from January 1, 2023 through March 31, 2026\. That gives us a complete, bounded hourly weather sidecar that is ready for downstream harmonization and app use\.

With the staged weather frame validated, we can write the final deliverables for downstream use\. This export step saves the study\-period station observations, the station manifest, a compact consolidated QA summary, and a lightweight markdown contract that records the grain, key fields, study window, and handoff notes for future notebooks and the eventual app\.

In [34]:
# ---------------------------------------------------------------------
# 1.1.7.5.2 Export staged weather outputs, QA summary, and contract
# ---------------------------------------------------------------------

def stringify_contract_value(value):
    """Format contract values for markdown output."""
    if pd.isna(value):
        return "NA"
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    return str(value)

for output_dir in [WEATHER_STAGED_DIR, WEATHER_QA_DIR, WEATHER_MANIFEST_DIR]:
    output_dir.mkdir(parents=True, exist_ok=True)

qa_summary_frames = []

coverage_overall_tidy_df = weather_qa_coverage_overall_df.rename(
    columns={"metric": "metric_name", "value": "metric_value"}
).copy()
coverage_overall_tidy_df.insert(0, "qa_section", "coverage_overall")
coverage_overall_tidy_df.insert(1, "entity_type", "notebook")
coverage_overall_tidy_df.insert(2, "entity_id", "all_stations")
coverage_overall_tidy_df.insert(3, "entity_name", "all_stations")
qa_summary_frames.append(
    coverage_overall_tidy_df[
        ["qa_section", "entity_type", "entity_id", "entity_name", "metric_name", "metric_value"]
    ]
)

station_metric_columns = [
    column
    for column in [
        "raw_row_count",
        "study_distinct_observation_hours",
        "coverage_pct_vs_study",
        "gap_count_over_1_hour",
        "max_gap_hours",
        "qa_status",
    ]
    if column in weather_qa_station_coverage_df.columns
]
station_coverage_tidy_df = weather_qa_station_coverage_df.melt(
    id_vars=["weather_station_id", "weather_station_name"],
    value_vars=station_metric_columns,
    var_name="metric_name",
    value_name="metric_value",
)
station_coverage_tidy_df.insert(0, "qa_section", "station_coverage")
station_coverage_tidy_df.insert(1, "entity_type", "station")
station_coverage_tidy_df = station_coverage_tidy_df.rename(
    columns={
        "weather_station_id": "entity_id",
        "weather_station_name": "entity_name",
    }
)
qa_summary_frames.append(
    station_coverage_tidy_df[
        ["qa_section", "entity_type", "entity_id", "entity_name", "metric_name", "metric_value"]
    ]
)

duplicate_summary_tidy_df = weather_qa_duplicate_summary_df.rename(
    columns={"metric": "metric_name", "value": "metric_value"}
).copy()
duplicate_summary_tidy_df.insert(0, "qa_section", "duplicate_summary")
duplicate_summary_tidy_df.insert(1, "entity_type", "notebook")
duplicate_summary_tidy_df.insert(2, "entity_id", "all_stations")
duplicate_summary_tidy_df.insert(3, "entity_name", "all_stations")
qa_summary_frames.append(
    duplicate_summary_tidy_df[
        ["qa_section", "entity_type", "entity_id", "entity_name", "metric_name", "metric_value"]
    ]
)

gap_metric_columns = [
    column
    for column in [
        "duplicate_station_hour_rows",
        "distinct_observation_hours",
        "gap_count_over_1_hour",
        "max_gap_hours",
        "min_gap_hours",
    ]
    if column in weather_qa_gap_summary_df.columns
]
gap_summary_tidy_df = weather_qa_gap_summary_df.melt(
    id_vars=["weather_station_id", "weather_station_name"],
    value_vars=gap_metric_columns,
    var_name="metric_name",
    value_name="metric_value",
)
gap_summary_tidy_df.insert(0, "qa_section", "gap_summary")
gap_summary_tidy_df.insert(1, "entity_type", "station")
gap_summary_tidy_df = gap_summary_tidy_df.rename(
    columns={
        "weather_station_id": "entity_id",
        "weather_station_name": "entity_name",
    }
)
qa_summary_frames.append(
    gap_summary_tidy_df[
        ["qa_section", "entity_type", "entity_id", "entity_name", "metric_name", "metric_value"]
    ]
)

measurement_metric_columns = [
    column
    for column in [
        "present",
        "row_count",
        "null_count",
        "null_pct",
        "distinct_non_null_value_count",
        "coerced_numeric_null_count",
        "min_value",
        "median_value",
        "max_value",
        "sentinel_value_count",
        "out_of_bounds_count",
        "top_non_null_values",
    ]
    if column in weather_qa_measurement_missingness_df.columns
]
measurement_tidy_df = weather_qa_measurement_missingness_df.melt(
    id_vars=["measurement_label", "source_column"],
    value_vars=measurement_metric_columns,
    var_name="metric_name",
    value_name="metric_value",
)
measurement_tidy_df.insert(0, "qa_section", "measurement_missingness")
measurement_tidy_df.insert(1, "entity_type", "measurement")
measurement_tidy_df = measurement_tidy_df.rename(
    columns={
        "measurement_label": "entity_id",
        "source_column": "entity_name",
    }
)
qa_summary_frames.append(
    measurement_tidy_df[
        ["qa_section", "entity_type", "entity_id", "entity_name", "metric_name", "metric_value"]
    ]
)

weather_hourly_qa_summary_df = pd.concat(qa_summary_frames, ignore_index=True)

contract_lines = [
    "# Weather Data Contract",
    "",
    "## Output Paths",
    "",
    f"- Curated hourly observations: `{WEATHER_HOURLY_OUTPUT_PATH.as_posix()}`",
    f"- Optional full audit observations: `{WEATHER_HOURLY_AUDIT_OUTPUT_PATH.as_posix()}`",
    f"- Station manifest: `{WEATHER_STATION_MANIFEST_PATH.as_posix()}`",
    f"- QA summary: `{WEATHER_HOURLY_QA_SUMMARY_PATH.as_posix()}`",
    f"- Data contract: `{WEATHER_DATA_CONTRACT_PATH.as_posix()}`",
    "",
    "## Notebook Scope",
    "",
    "- Source family: NOAA / GHCNh station-year weather observations",
    "- The main export is a curated hourly weather feature handoff rather than a full raw-source dump.",
    "- A broader study-period audit export can be written separately when needed for debugging or additional feature engineering.",
    "",
    "## Intended Grain",
    "",
    "- `weather_hourly_observations.parquet`: one row per retained station observation timestamp within the study period",
    "- `weather_hourly_observations_full_audit.parquet`: optional broader audit export at the same timestamp grain",
    "- `weather_station_manifest.parquet`: one row per observed weather station",
    "- `weather_hourly_qa_summary.parquet`: one row per QA metric",
    "",
    "## Primary Keys",
    "",
    "- Observations: `weather_station_id`, `observation_datetime`",
    "- Station manifest: `weather_station_id`",
    "- QA summary: `qa_section`, `entity_type`, `entity_id`, `metric_name`",
    "",
    "## Curated Export Columns",
    "",
    "- `weather_station_id`",
    "- `weather_station_name`",
    "- `observation_datetime`",
    "- `observation_date`",
    "- `observation_hour`",
    "- `temperature`",
    "- `precipitation`",
    "- `visibility`",
    "- `wind_speed`",
    "- `wind_gust`",
    "- `relative_humidity`",
    "- `ceiling_height`",
    "- `pressure_3hr_change`",
    "",
    "## Study Window",
    "",
    f"- Study start: `{STUDY_START_TIMESTAMP.date()}`",
    f"- Study end: `{(STUDY_END_TIMESTAMP_EXCLUSIVE - pd.Timedelta(seconds=1)).date()}`",
    "",
    "## Export Summary",
    "",
    f"- Curated study-period rows: `{len(weather_hourly_export_df)}`",
    f"- Curated study-period stations: `{weather_hourly_export_df['weather_station_id'].nunique()}`",
    f"- Min observation timestamp: `{stringify_contract_value(weather_hourly_export_df['observation_datetime'].min())}`",
    f"- Max observation timestamp: `{stringify_contract_value(weather_hourly_export_df['observation_datetime'].max())}`",
    "",
    "## Known Caveats",
    "",
    "- High null rates for some weather fields can reflect source sparsity, optional report types, or event-driven measurements rather than broken ingestion.",
    "- `wind_direction`, pressure levels, and raw present-weather codes are excluded from the curated export and remain available only through the optional audit export.",
    "- Sentinel values such as `999` should still be cleaned downstream before modeling if the broader audit fields are reused.",
]

weather_data_contract_md = "\n".join(contract_lines)

if SAVE_WEATHER_OUTPUTS and (
    FORCE_REBUILD_WEATHER_OUTPUTS
    or should_build(WEATHER_HOURLY_OUTPUT_PATH)
    or should_build(WEATHER_STATION_MANIFEST_PATH)
    or should_build(WEATHER_HOURLY_QA_SUMMARY_PATH)
    or should_build(WEATHER_DATA_CONTRACT_PATH)
):
    print_step("Writing curated staged weather outputs.")
    weather_hourly_export_df.to_parquet(WEATHER_HOURLY_OUTPUT_PATH, index=False)
    weather_station_manifest_df.to_parquet(WEATHER_STATION_MANIFEST_PATH, index=False)
    weather_hourly_qa_summary_df.to_parquet(WEATHER_HOURLY_QA_SUMMARY_PATH, index=False)
    WEATHER_DATA_CONTRACT_PATH.write_text(weather_data_contract_md, encoding="utf-8")
else:
    print_step("Skipping curated weather writes because outputs already exist and rebuild is not forced.")

if WRITE_FULL_WEATHER_AUDIT_EXPORT and (
    FORCE_REBUILD_WEATHER_OUTPUTS or should_build(WEATHER_HOURLY_AUDIT_OUTPUT_PATH)
):
    print_step("Writing optional full weather audit export.")
    weather_hourly_audit_export_df.to_parquet(WEATHER_HOURLY_AUDIT_OUTPUT_PATH, index=False)

final_weather_handoff_summary_df = pd.DataFrame(
    {
        "artifact": [
            "weather_hourly_observations",
            "weather_hourly_observations_full_audit",
            "weather_station_manifest",
            "weather_hourly_qa_summary",
            "weather_data_contract",
        ],
        "path": [
            WEATHER_HOURLY_OUTPUT_PATH.as_posix(),
            WEATHER_HOURLY_AUDIT_OUTPUT_PATH.as_posix(),
            WEATHER_STATION_MANIFEST_PATH.as_posix(),
            WEATHER_HOURLY_QA_SUMMARY_PATH.as_posix(),
            WEATHER_DATA_CONTRACT_PATH.as_posix(),
        ],
        "exists_after_run": [
            WEATHER_HOURLY_OUTPUT_PATH.exists(),
            WEATHER_HOURLY_AUDIT_OUTPUT_PATH.exists(),
            WEATHER_STATION_MANIFEST_PATH.exists(),
            WEATHER_HOURLY_QA_SUMMARY_PATH.exists(),
            WEATHER_DATA_CONTRACT_PATH.exists(),
        ],
    }
)

weather_qa_export_overview_df = (
    weather_hourly_qa_summary_df.groupby("qa_section", dropna=False)
    .size()
    .reset_index(name="qa_metric_rows")
)

display(final_weather_handoff_summary_df)
display(weather_qa_export_overview_df)

print_step(
    f"Final curated weather handoff complete for {weather_hourly_export_df['weather_station_id'].nunique()} stations and {len(weather_hourly_export_df):,} study-period observations."
)

[1.1.7] Skipping curated weather writes because outputs already exist and rebuild is not forced.


,artifact,path,exists_after_run
0,weather_hourly_observations,pipeline_data/1.1.7.staged_weather/weather_hourly_observations.parquet,True
1,weather_hourly_observations_full_audit,pipeline_data/1.1.7.staged_weather/weather_hourly_observations_full_audit.parquet,False
2,weather_station_manifest,pipeline_data/1.1.7.staged_weather/weather_station_manifest.parquet,True
3,weather_hourly_qa_summary,pipeline_data/1.1.7.qa/weather_hourly_qa_summary.parquet,True
4,weather_data_contract,pipeline_data/1.1.7.manifests/weather_data_contract.md,True


,qa_section,qa_metric_rows
0,coverage_overall,7
1,duplicate_summary,3
2,gap_summary,20
3,measurement_missingness,300
4,station_coverage,12


[1.1.7] Final curated weather handoff complete for 4 stations and 221,939 study-period observations.


Findings: The exported QA summary now looks structurally clean and well\-partitioned by topic rather than collapsing everything into one noisy mixed table\. The section counts make sense: a small notebook\-level coverage summary, a compact duplicate summary, station\-level coverage and gap diagnostics, and a much larger measurement\-missingness section driven by field\-level weather QA\. That measurement\_missingness = 300 count is not automatically a problem; it is just the product of multiple weather measures crossed with multiple QA metrics per measure\. In other words, this reads like a tidy long\-format QA export, not a broken one\.

## Close

This notebook stages NOAA / GHCNh hourly weather observations into a clean study\-period sidecar that downstream notebooks can use without revisiting the raw source files\. We now have validated station\-level hourly exports, a compact station manifest, and QA outputs that make the timestamp coverage, missingness patterns, and field availability easy to audit\. Just as importantly, the notebook keeps weather in the right lane: loaded and documented here, with zone assignment and multimodal integration handled later in the pipeline\.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=5c43663c-345d-4c21-9332-dd4f928af3ba' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>